# NephroLLM — Execution Plan

1. Environment setup
2. Load Llama 3.2 3B Instruct
3. Define & Verify the base-model Inference
4. Load and validate the nephrology benchmark
5. Run a fresh baseline evaluation
6. Analyze baseline failures
7. Prepare fine-tuning dataset
8. Fine-tune using QLoRA
9. Save the LoRA adapter
10. Run the exact same evaluation on the fine-tuned model
11. Evaluate on held-out questions
12. Compare base vs fine-tuned model
13. Document findings, limitations, and conclusions

# NephroLLM — Fine-Tuning Llama 3.2 3B for Nephrology Q&A

Educational fine-tuning experiment using QLoRA to adapt Llama 3.2 3B Instruct to nephrology-focused questions.

**Objective:** Compare the base model against the QLoRA fine-tuned model using the same evaluation benchmark and evaluation protocol.

# 1. Environment Setup - Install Required Libraries

This cell installs the libraries required for model loading, inference, dataset handling, quantization, and parameter-efficient fine-tuning.

We use:

- **Transformers** - load and run Llama 3.2 3B Instruct
- **Datasets** - prepare and manage training/evaluation data
- **Accelerate** - support efficient model execution on the available GPU
- **PEFT** - implement LoRA/QLoRA fine-tuning
- **BitsAndBytes** - enable 4-bit quantization
- **TRL** - provide utilities for supervised fine-tuning

The goal at this stage is only to prepare the environment. No model weights are modified or fine-tuned in this cell.

# 1.1 Install required libraries

In [1]:
# cell 2
!pip install -U transformers datasets accelerate peft "bitsandbytes>=0.46.1" trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 73.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 78.5 MB/s eta 0:00:00:00:01
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenize

# 1.2 Verify environment and GPU
Environment and Hardware Verification

Before loading the model, we verify the versions of the main libraries and confirm that CUDA and the Kaggle GPU are available.

This is important because NephroLLM uses 4-bit quantization and will later use QLoRA for parameter-efficient fine-tuning. Both depend on compatible versions of the deep-learning and quantization libraries and access to a CUDA-enabled GPU.

We also record the available GPU memory so that the later fine-tuning configuration can be chosen with the available hardware in mind.

**Expected outcome:** CUDA should be available and a GPU should be detected.

In [4]:
import torch
import transformers
import accelerate
import bitsandbytes
import peft

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("PEFT:", peft.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
Transformers: 5.17.0
Accelerate: 1.15.0
BitsAndBytes: 0.50.2
PEFT: 0.21.0

CUDA available: True
GPU: Tesla T4


# 1.3 Authenticate with Hugging Face

Llama 3.2 3B Instruct is accessed through Hugging Face and requires an authenticated account with access to the model.

This cell retrieves the Hugging Face access token from Kaggle Secrets rather than hard-coding credentials in the notebook.

We only verify that a token is available. The token itself is never printed.

**Why this matters:** keeping credentials in Kaggle Secrets prevents sensitive authentication information from being committed to the notebook or exposed in the GitHub repository.

In [5]:
# cell 6
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HF_TOKEN")

print("HF token available:", hf_token is not None)
print("Token length:", len(hf_token) if hf_token else 0)

HF token available: True
Token length: 37


# 2 - Load Llama 3.2 3B Instruct

### Loading the Base Model with 4-bit Quantization

We now load the pretrained **Llama 3.2 3B Instruct** model that will serve as the starting point for the NephroLLM experiment.

The model is loaded using **4-bit NF4 quantization** with double quantization. This reduces the GPU memory required to load the 3B-parameter model and makes the later QLoRA experiment practical on the available Kaggle GPU.

At this stage, the model is still completely **untuned**. This is our base model and will be evaluated before any nephrology-specific fine-tuning.

**Configuration**

- **Base model:** `meta-llama/Llama-3.2-3B-Instruct`
- **Quantization:** 4-bit
- **Quantization type:** NF4
- **Compute dtype:** float16
- **Double quantization:** enabled
- **Device mapping:** automatic

**Why 4-bit quantization?**

QLoRA combines low-bit quantization of the base model with LoRA adapters that contain the trainable parameters. This allows us to fine-tune a large language model while using substantially less GPU memory than full-parameter fine-tuning.

**Expected outcome:** the tokenizer and quantized base model should load successfully on the available GPU.

In [6]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Retrieve Hugging Face token from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Authenticate with Hugging Face
login(token=hf_token, add_to_git_credential=False)

print("Hugging Face authentication successful.")
print("Token available:", bool(hf_token))

Hugging Face authentication successful.
Token available: True


In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=hf_token
)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

model.eval()

print("\nModel loaded successfully.")
print("Model device:", model.device)
print("Model dtype:", model.dtype)

Loading tokenizer...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]


Model loaded successfully.
Model device: cuda:0
Model dtype: torch.bfloat16


# Experimental Design

### Base Model → Fine-Tuning → Re-evaluation

The goal of NephroLLM is to measure whether nephrology-specific fine-tuning improves the model's performance on multiple-choice nephrology questions.

The experiment follows a controlled evaluation design:

1. **Base model evaluation**  
   Run the untuned Llama 3.2 3B Instruct model on the **636-question primary evaluation benchmark**.

2. **Baseline error analysis**  
   Analyze incorrect answers, non-answers, response formatting, and performance across nephrology subjects.

3. **Training data preparation**  
   Prepare a **separate nephrology training corpus** that is not part of the 636-question primary benchmark or the 160-question held-out benchmark.

4. **QLoRA fine-tuning**  
   Fine-tune the base model using parameter-efficient QLoRA rather than updating all model parameters.

5. **Post-training evaluation**  
   Run the fine-tuned model on the **exact same 636 primary questions** using the same evaluation protocol.

6. **Held-out evaluation**  
   Evaluate the fine-tuned model separately on **160 held-out nephrology questions** that were not used for training.

7. **Comparison**  
   Compare base-model and fine-tuned performance using the same scoring methodology.

### Evaluation Split

| Split | Questions | Purpose |
|---|---:|---|
| Primary evaluation | 636 | Base vs fine-tuned comparison |
| Held-out evaluation | 160 | Generalization check |
| **Total evaluation data** | **796** | Evaluation only |

**Important:** the 636 primary questions and 160 held-out questions are evaluation data. They must not be used as QLoRA training examples.

This separation is important because using evaluation questions for training would introduce data leakage and make the before/after comparison unreliable.

**Expected outcome:** establish a reproducible experimental protocol where the only intended change between the baseline and post-training evaluation is the model's fine-tuned parameters.

# 3 - Define & Verify the base Model Inference 

# 3.1 Define standardized inference

Before running the benchmark, we define a reusable inference function that sends questions to the model using the Llama 3.2 Instruct chat format.

The function applies the model's chat template, generates a deterministic response, and returns only the newly generated text rather than the original prompt.

### Generation protocol

- **Sampling:** disabled (`do_sample=False`)
- **Maximum new tokens:** configurable per evaluation
- **Padding:** uses the tokenizer's EOS token
- **Inference mode:** gradients disabled with `torch.no_grad()`

Disabling sampling is important for evaluation because it makes the model's responses deterministic under the same configuration. This allows the base-model and fine-tuned evaluations to use the same generation protocol.

The function will be reused throughout the experiment rather than creating separate inference implementations for different stages.

**Expected outcome:** a single standardized inference function that can be used for smoke tests, benchmark evaluation, and post-training evaluation.

In [7]:
# cell 11
def ask_model(question, max_new_tokens=256):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an educational assistant specializing in nephrology. "
                "Provide clear, factual, educational explanations. "
                "Do not provide diagnosis or personalized medical advice."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

# 3.2 Run an inference smoke test

Before running the full benchmark, we perform a small smoke test using a simple nephrology question.

The purpose of this test is **not to measure model performance**. It verifies that the complete inference pipeline is working correctly:

- the tokenizer can format the prompt,
- the model can generate a response,
- the GPU inference path is functioning,
- the generated tokens can be decoded correctly, and
- the `ask_model()` function returns usable text.

This provides a quick sanity check before starting the longer 636-question baseline evaluation.

**Expected outcome:** the model should return a coherent educational response to the test question.

In [8]:
# cell 13
question = "What is the primary function of the glomerulus?"

answer = ask_model(question)

print("QUESTION:")
print(question)

print("\nMODEL ANSWER:")
print(answer)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


QUESTION:
What is the primary function of the glomerulus?

MODEL ANSWER:
The primary function of the glomerulus is to filter waste and excess fluids from the blood. It is a critical component of the nephron, the functional unit of the kidney, and plays a vital role in maintaining overall kidney function.

The glomerulus is a tuft of capillaries that receives blood from the afferent arteriole and sends it to the efferent arteriole. As the blood flows through the glomerulus, it is subjected to a pressure gradient that forces the fluid to pass through the glomerular basement membrane and into the Bowman's capsule. This process is known as filtration.

The glomerulus filters out waste products, excess ions, and excess water from the blood, while retaining essential nutrients, proteins, and other vital substances. The filtered waste products, including urea, creatinine, and other waste products, are then transported to the renal tubules for further processing and excretion.

In essence, the

# 4. Load and validate the Nephrology Benchmark

# 4.1 Experimental and data strategy


### Separating Training Data from Evaluation Data

A key requirement of this experiment is to keep the evaluation questions completely separate from the fine-tuning data.

The NephroLLM benchmark currently provides two evaluation splits:

- **Primary evaluation:** 636 questions
- **Held-out evaluation:** 160 questions

There is **no training split in this benchmark**. Therefore, these 796 questions will not be used as QLoRA training examples.

### Role of Each Dataset

| Dataset | Size | Role |
|---|---:|---|
| Primary evaluation | 636 | Base-model baseline and post-training comparison |
| Held-out evaluation | 160 | Generalization evaluation after fine-tuning |
| Separate training corpus | TBD | QLoRA fine-tuning |

The separate training corpus must contain nephrology educational material that is independent of these evaluation questions.

### Why this separation matters

If benchmark questions were included in the fine-tuning data, the model could memorize questions or answers rather than demonstrate genuine improvement in nephrology knowledge.

Keeping the evaluation data unseen during training allows us to make a cleaner before-versus-after comparison and provides a more meaningful test of generalization.

**Important:** at this stage, the evaluation benchmark is available, but the separate QLoRA training corpus still needs to be prepared.

**Expected outcome:** establish a clear data boundary before beginning fine-tuning and prevent evaluation-data leakage.

# 4.2 Load the primary evaluation benchmark

### Evaluating the Base Model on the Primary Benchmark

We now evaluate the untuned Llama 3.2 3B Instruct model on the **636-question primary nephrology benchmark**.

This establishes the quantitative baseline that will later be compared with the QLoRA fine-tuned model.

For every question, we preserve the model's raw response along with the parsed answer and ground-truth answer. This allows us to distinguish between:

- an incorrect answer,
- a response that could not be parsed,
- and a genuine non-answer or refusal.

### Evaluation protocol

The same protocol will be reused after fine-tuning:

- **Questions:** 636 primary evaluation questions
- **Model:** Llama 3.2 3B Instruct
- **Sampling:** deterministic (`do_sample=False`)
- **Maximum new tokens:** 96
- **Prompt format:** standardized multiple-choice prompt
- **Output:** raw model response + extracted answer + ground truth
- **Scoring:** exact option-level accuracy

The baseline results will serve as the reference point for measuring any change after QLoRA fine-tuning.

**Expected outcome:** a complete baseline prediction file containing one evaluation record for each of the 636 benchmark questions.

In [9]:
# cell 16
import os
import pandas as pd

# Find the finalized primary evaluation file
matches = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "nephrollm_primary_eval.csv":
            matches.append(os.path.join(root, file))

print("Found:", matches)

if len(matches) != 1:
    raise ValueError(
        f"Expected exactly one primary evaluation file, found: {matches}"
    )

eval_path = matches[0]
eval_df = pd.read_csv(eval_path)

print("Evaluation file:", eval_path)
print("Questions loaded:", len(eval_df))

display(
    eval_df[
        ["BenchmarkID", "Subject", "Question", "Choices", "Answer"]
    ].head()
)

Found: ['/kaggle/input/datasets/shantanuss/nejm-ai-nephrology-benchmark/nephrollm_primary_eval.csv']
Evaluation file: /kaggle/input/datasets/shantanuss/nejm-ai-nephrology-benchmark/nephrollm_primary_eval.csv
Questions loaded: 636


,BenchmarkID,Subject,Question,Choices,Answer
0,NEPHRO-EVAL-0001,Acute Kidney Injury & Critical Care Nephrology,Question 2824:\nThe transfusion of longer-term...,1. Increased risk of transfusion-related react...,B
1,NEPHRO-EVAL-0002,Acute Kidney Injury & Critical Care Nephrology,"Based on data from the NUTRIREA-2 study, enter...",A. Decreased incidence of systemic bacteremia\...,B
2,NEPHRO-EVAL-0003,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following is CORRECT about th...,1. She is at decreased risk for inpatient mort...,B
3,NEPHRO-EVAL-0004,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following is associated with ...,A. A decreased risk of surgical site infection...,B
4,NEPHRO-EVAL-0005,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following is the MOST importa...,A. Administration of a bolus of 30 ml/kg 0.9% ...,D


# 5. Run a Fresh Baseline Evaluation

# 5.1 Lock the evaluation protocol
Locking the Evaluation Configuration

Before running the full baseline, we explicitly define the evaluation configuration.

This is important because the same evaluation protocol must be used later when testing the QLoRA fine-tuned model. Keeping the protocol fixed makes the before-versus-after comparison reproducible.

For each question, we will preserve:

1. The original benchmark question
2. The available answer choices
3. The model's raw response
4. The answer extracted by the evaluator
5. The ground-truth answer
6. Whether the extracted answer is correct

### Fixed evaluation settings

| Setting | Configuration |
|---|---|
| Evaluation split | Primary |
| Number of questions | 636 |
| Generation | Deterministic |
| `do_sample` | `False` |
| `max_new_tokens` | `96` |
| Scoring | Exact option match |
| Raw responses | Saved |
| Parsed answers | Saved |
| Ground truth | Saved |

The evaluator is intentionally conservative: if the model does not provide a recognizable answer choice, the response is treated as **unparsed/non-answer** rather than guessing what the model intended.

**Expected outcome:** the evaluation configuration is clearly documented before the baseline results are generated.

# 5.2 Define the MCQ answer parser

Multiple-Choice Answer Evaluator

The benchmark contains multiple-choice questions with four or five answer options. The model may express its answer in different formats, such as:

- `ANSWER: A`
- `ANSWER: 2`
- `C`
- `Option D`
- `Choice: B`

The evaluator normalizes these responses into a standard option label (`A`–`E`) before comparing them with the benchmark's ground-truth answer.

### Conservative parsing

The parser prioritizes explicit answer statements and recognizable option formats.

It does **not** attempt to infer an answer from general prose when the model does not clearly select an option.

For example, if the model produces a refusal, uncertainty statement, or explanation without a clearly identifiable answer choice, the response is recorded as **unparsed/non-answer**.

This conservative approach is intentional. Guessing the model's intended answer could artificially inflate the measured accuracy.

### Scoring

For each question:

```text
Predicted option == Ground-truth option
            ↓
        Correct / Incorrect

In [10]:
import re

def extract_answer(response):
    """
    Extract a single answer choice.

    Supports alphabetical and numerical answer formats.
    Numerical choices are mapped by position:
    1=A, 2=B, 3=C, 4=D, 5=E
    """

    if not response:
        return None

    text = str(response).strip().upper()

    # 1. Explicit ANSWER / CORRECT ANSWER formats
    patterns = [
        r"\bANSWER\s*:\s*([A-E])\b",
        r"\bCORRECT ANSWER\s*(?:IS)?\s*:?\s*([A-E])\b",
        r"\bTHE ANSWER\s*(?:IS)?\s*:?\s*([A-E])\b",
        r"\bMY ANSWER\s*(?:IS)?\s*:?\s*([A-E])\b",
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1)

    # 2. Explicit numeric ANSWER format
    numeric_patterns = [
        r"\bANSWER\s*:\s*([1-5])\b",
        r"\bCORRECT ANSWER\s*(?:IS)?\s*:?\s*([1-5])\b",
        r"\bTHE ANSWER\s*(?:IS)?\s*:?\s*([1-5])\b",
        r"\bMY ANSWER\s*(?:IS)?\s*:?\s*([1-5])\b",
    ]

    for pattern in numeric_patterns:
        match = re.search(pattern, text)
        if match:
            return "ABCDE"[int(match.group(1)) - 1]

    # 3. Standalone answer formats: A, A., A)
    lines = [x.strip() for x in text.splitlines() if x.strip()]

    for line in lines[:8]:
        match = re.fullmatch(r"([A-E])[\.\)]?", line)
        if match:
            return match.group(1)

    # 4. Standalone numeric answer: 1, 1., 1)
    for line in lines[:8]:
        match = re.fullmatch(r"([1-5])[\.\)]?", line)
        if match:
            return "ABCDE"[int(match.group(1)) - 1]

    # 5. OPTION / CHOICE format
    for line in lines[:8]:
        match = re.fullmatch(
            r"(?:OPTION|CHOICE)\s*[:\-]?\s*([A-E])[\.\)]?",
            line
        )
        if match:
            return match.group(1)

        match = re.fullmatch(
            r"(?:OPTION|CHOICE)\s*[:\-]?\s*([1-5])[\.\)]?",
            line
        )
        if match:
            return "ABCDE"[int(match.group(1)) - 1]

    return None

# 5.3 Define the standardized MCQ prompt

We now define the prompt used to evaluate each benchmark question.

The model is explicitly instructed to select **one answer from the provided choices** and return it in a standardized format:

```text
ANSWER: <option letter>

In [11]:
# cell 19
def ask_mcq_fast(question, choices, max_new_tokens=32):
    """
    Ask the base model to answer one MCQ.

    The benchmark may contain 4 or 5 choices and may use
    alphabetical or numerical labels.
    """

    choice_lines = str(choices).strip()

    prompt = f"""Answer this multiple-choice nephrology question.

Question:
{question}

Choices:
{choice_lines}

Select the single best answer from the choices provided.

Return ONLY:
ANSWER: <option letter>
"""

    return ask_model(
        prompt,
        max_new_tokens=max_new_tokens
    )

# 5.4 Run a small evaluation sanity check


In [12]:
# cell 27 
test_df = eval_df.head(5).copy()

test_results = []

for _, row in test_df.iterrows():

    response = ask_mcq_fast(
        row["Question"],
        row["Choices"]
    )

    predicted = extract_answer(response)
    expected = str(row["Answer"]).strip().upper()

    test_results.append({
        "BenchmarkID": row["BenchmarkID"],
        "Expected": expected,
        "Predicted": predicted,
        "Correct": predicted == expected,
        "ModelResponse": response
    })

test_results_df = pd.DataFrame(test_results)

display(test_results_df)

,BenchmarkID,Expected,Predicted,Correct,ModelResponse
0,NEPHRO-EVAL-0001,B,D,False,D
1,NEPHRO-EVAL-0002,B,A,False,A
2,NEPHRO-EVAL-0003,B,A,False,A.
3,NEPHRO-EVAL-0004,B,B,True,B
4,NEPHRO-EVAL-0005,D,D,True,D


# 5.5 Run the full 636-question baseline

### Running the 636-Question Benchmark

The evaluation protocol is now locked, so we can run the complete primary benchmark against the untuned base model.

The benchmark contains **636 questions**. Because generation can be time-consuming on the available GPU, the evaluation is executed in batches and each batch is saved separately.

For every question, we retain the model's raw response and the information required for later scoring and error analysis.

### Why save batches?

Saving intermediate results provides two benefits:

1. **Fault tolerance:** if a long evaluation run is interrupted, completed batches do not need to be regenerated.
2. **Reproducibility:** each batch produces a persistent prediction file that can be independently inspected and aggregated.

After all batches are complete, the individual files are combined into one final baseline result set containing all 636 questions.

**Important:** this run is the base-model baseline. No QLoRA training has been performed yet.

**Expected outcome:** 636 baseline predictions are generated and stored for subsequent scoring and analysis.

In [13]:
from tqdm.auto import tqdm
import os
import pandas as pd

BATCH_SIZE = 50

BASELINE_PREFIX = "/kaggle/working/nephrollm_baseline_final_batch_"

for start in range(0, len(eval_df), BATCH_SIZE):

    end = min(start + BATCH_SIZE, len(eval_df))
    batch = eval_df.iloc[start:end].copy()

    batch_number = start // BATCH_SIZE + 1

    print(
        f"\nRunning Batch {batch_number}: "
        f"questions {start + 1} to {end}"
    )

    batch_results = []

    for _, row in tqdm(
        batch.iterrows(),
        total=len(batch),
        desc=f"Batch {batch_number}"
    ):

        response = ask_mcq_fast(
            row["Question"],
            row["Choices"]
        )

        predicted = extract_answer(response)
        expected = str(row["Answer"]).strip().upper()

        batch_results.append({
            "BenchmarkID": row["BenchmarkID"],
            "Subject": row["Subject"],
            "Question": row["Question"],
            "Choices": row["Choices"],
            "Expected": expected,
            "Predicted": predicted,
            "Correct": predicted == expected,
            "ModelResponse": response
        })

    batch_df = pd.DataFrame(batch_results)

    batch_path = (
        f"{BASELINE_PREFIX}"
        f"{start + 1:03d}_{end:03d}.csv"
    )

    batch_df.to_csv(batch_path, index=False)

    print(f"Saved: {batch_path}")
    print(
        f"Batch accuracy: "
        f"{batch_df['Correct'].mean():.2%}"
    )
    print(
        f"Unparsed: "
        f"{batch_df['Predicted'].isna().sum()}"
    )


Running Batch 1: questions 1 to 50


Batch 1:   0%|          | 0/50 [00:00<?, ?it/s]

Saved: /kaggle/working/nephrollm_baseline_final_batch_001_050.csv
Batch accuracy: 56.00%
Unparsed: 1

Running Batch 2: questions 51 to 100


Batch 2:   0%|          | 0/50 [00:00<?, ?it/s]

Saved: /kaggle/working/nephrollm_baseline_final_batch_051_100.csv
Batch accuracy: 48.00%
Unparsed: 1

Running Batch 3: questions 101 to 150


...

Saved: /kaggle/working/nephrollm_baseline_final_batch_501_550.csv
Batch accuracy: 28.00%
Unparsed: 5

Running Batch 12: questions 551 to 600


Batch 12:   0%|          | 0/50 [00:00<?, ?it/s]

Saved: /kaggle/working/nephrollm_baseline_final_batch_551_600.csv
Batch accuracy: 44.00%
Unparsed: 2

Running Batch 13: questions 601 to 636


Batch 13:   0%|          | 0/36 [00:00<?, ?it/s]

Saved: /kaggle/working/nephrollm_baseline_final_batch_601_636.csv
Batch accuracy: 33.33%
Unparsed: 1


# 5.6 Aggregate baseline predictions

The 636-question baseline was executed in multiple batches. We now combine those batch outputs into a single evaluation dataset.

Before calculating the final score, we verify that:

- all expected batch files are present,
- the combined result contains 636 rows,
- each benchmark question appears only once, and
- the final result can be saved as a single reproducible CSV.

This step converts the raw batch predictions into the canonical baseline result file that will be used for all subsequent analysis.

**Expected outcome:** one consolidated baseline file containing exactly **636 unique benchmark questions**.

In [14]:
import glob
import pandas as pd

baseline_files = sorted(
    glob.glob(
        "/kaggle/working/nephrollm_baseline_final_batch_*.csv"
    )
)

print("Fresh baseline batch files:", len(baseline_files))

if len(baseline_files) != 13:
    raise ValueError(
        f"Expected 13 fresh baseline files, found {len(baseline_files)}"
    )

baseline_df = pd.concat(
    [pd.read_csv(f) for f in baseline_files],
    ignore_index=True
)

# Safety checks
print("Rows before deduplication:", len(baseline_df))
print(
    "Unique BenchmarkIDs:",
    baseline_df["BenchmarkID"].nunique()
)

if len(baseline_df) != 636:
    raise ValueError(
        f"Expected 636 rows, found {len(baseline_df)}"
    )

if baseline_df["BenchmarkID"].nunique() != 636:
    raise ValueError("Duplicate or missing BenchmarkIDs detected.")

# Save one consolidated fresh baseline file
baseline_path = (
    "/kaggle/working/"
    "nephrollm_baseline_final_results.csv"
)

baseline_df.to_csv(
    baseline_path,
    index=False
)

print("\nFresh baseline saved:")
print(baseline_path)

print("\nColumns:")
print(list(baseline_df.columns))

Fresh baseline batch files: 13
Rows before deduplication: 636
Unique BenchmarkIDs: 636

Fresh baseline saved:
/kaggle/working/nephrollm_baseline_final_results.csv

Columns:
['BenchmarkID', 'Subject', 'Question', 'Choices', 'Expected', 'Predicted', 'Correct', 'ModelResponse']


# 6. Analyze Baseline Failures
### Measuring Base-Model Performance

With all 636 predictions consolidated, we now calculate the baseline performance of the untuned Llama 3.2 3B Instruct model.

We report two complementary metrics:

- **Accuracy (all questions):** correct answers divided by all 636 benchmark questions.
- **Accuracy (answered-only):** correct answers divided by questions for which the evaluator successfully extracted a valid answer choice.

We also report the number of **unparsed/non-answer responses** separately. This is important because a model can fail either by selecting the wrong option or by failing to provide a usable answer.

### Fresh baseline result

| Metric | Result |
|---|---:|
| Questions evaluated | ** ** |
| Correct | ** ** |
| Incorrect | ** ** |
| Unparsed / non-answer | ** ** |
| Accuracy — all questions | ** ** |
| Accuracy — answered only | ** ** |

The **41.51%** figure is the primary baseline metric because it treats a non-answer as a failure on the benchmark item rather than excluding it from evaluation.

The answered-only figure is reported as a secondary diagnostic metric to show performance among responses where the evaluator could identify a selected option.

These baseline results will become the reference point for the post-QLoRA evaluation.

**Important:** these are the results from the fresh 636-question baseline run and the finalized conservative answer parser. No fine-tuning has been performed yet.

# 6.1 Calculate the initial baseline results


In [15]:
print("NEPHROLLM — FRESH BASELINE")
print("==========================")

total_questions = len(baseline_df)
correct = baseline_df["Correct"].sum()
unparsed = baseline_df["Predicted"].isna().sum()
incorrect = total_questions - correct - unparsed

accuracy_all = correct / total_questions
accuracy_answered = (
    correct / (total_questions - unparsed)
    if total_questions - unparsed > 0
    else 0
)

print(f"Total questions:      {total_questions}")
print(f"Correct:              {correct}")
print(f"Incorrect:            {incorrect}")
print(f"Unparsed:             {unparsed}")
print(f"Accuracy (all):       {accuracy_all:.2%}")
print(f"Accuracy (answered):  {accuracy_answered:.2%}")

NEPHROLLM — FRESH BASELINE
Total questions:      636
Correct:              265
Incorrect:            328
Unparsed:             43
Accuracy (all):       41.67%
Accuracy (answered):  44.69%


# 6.2 Inspect unparsed responses

After applying the finalized answer parser, we inspect the responses that still cannot be mapped reliably to an answer choice.

The purpose of this step is to validate that the remaining parsing failures are genuine **non-answers or ambiguous responses**, rather than valid answer formats that the parser has missed.

We display a sample of the unparsed responses together with their expected answers and available choices. This allows us to manually inspect the model's actual output before treating these responses as failures in the final baseline score.

The unparsed responses are **not reinterpreted or guessed** during this inspection. If the model does not clearly select an answer option, the response remains unparsed.

**Expected outcome:** confirm that the remaining unparsed responses represent genuine non-answers or ambiguous outputs and that the conservative parser should remain unchanged.

In [16]:
unparsed_df = baseline_df[
    baseline_df["Predicted"].isna()
].copy()

print("Unparsed responses:", len(unparsed_df))

display(
    unparsed_df[
        [
            "BenchmarkID",
            "Expected",
            "Choices",
            "ModelResponse"
        ]
    ].head(20)
)

Unparsed responses: 43


,BenchmarkID,Expected,Choices,ModelResponse
24,NEPHRO-EVAL-0025,C,A. Intermittent hemodiaﬁltration\nB. Intermitt...,I'm happy to help you with your nephrology que...
64,NEPHRO-EVAL-0065,A,A. Regional citrate anticoagulation (RC...,I'm happy to help you with your nephrology que...
144,NEPHRO-EVAL-0145,D,A. Denosumab\nB. Increase calcium acetate\nC. ...,I'm happy to help you with your nephrology que...
155,NEPHRO-EVAL-0156,A,A. Cinacalcet therapy\nB. Calcitonin\nC. Estro...,I'm happy to help you with your nephrology que...
176,NEPHRO-EVAL-0177,C,1. Refer for cardiac catheterization\n2. Chest...,I'm happy to help you with your nephrology que...
180,NEPHRO-EVAL-0181,C,1. Ankylosing spondylitis\n2. Adynamic bone di...,I'm happy to help you with your nephrology que...
182,NEPHRO-EVAL-0183,D,1. Hypercalciuria\n2. Renal tubular acidosis\n...,I'm happy to help you with your nephrology que...
188,NEPHRO-EVAL-0189,C,1. A loss of function mutation in the calcium-...,I'm happy to help you with your nephrology que...
191,NEPHRO-EVAL-0192,A,1. Change calcium acetate to sevelamer carbona...,I can't provide a specific answer without know...
227,NEPHRO-EVAL-0228,D,A. Discontinue pravastatin now\nB. Discontinue...,I'm happy to help you with your nephrology que...


# Re-scoring the Complete Baseline

We now apply the finalized answer parser to all stored model responses and recompute the baseline metrics.

The scoring logic separates three outcomes:

- **Correct:** the parsed model answer matches the benchmark answer.
- **Incorrect:** the model selected an identifiable option, but it does not match the benchmark answer.
- **Unparsed:** the response does not contain a reliably identifiable answer choice.

Two accuracy measures are calculated:

- **Accuracy (all):** correct answers divided by all 636 questions.
- **Accuracy (answered):** correct answers divided by questions with a successfully parsed answer.

The re-scored results are saved to:

`nephrollm_baseline_final_results.csv`

This is a deterministic post-processing step. **No model inference is performed here**, so the underlying model responses remain exactly the same as those generated during the original baseline run.

**Expected outcome:** the finalized baseline metrics are produced using the locked conservative parser and the complete set of 636 stored responses.

In [17]:
baseline_df["Predicted"] = (
    baseline_df["ModelResponse"]
    .apply(extract_answer)
)

baseline_df["Correct"] = (
    baseline_df["Predicted"]
    == baseline_df["Expected"]
)

total_questions = len(baseline_df)
correct = baseline_df["Correct"].sum()
unparsed = baseline_df["Predicted"].isna().sum()
incorrect = total_questions - correct - unparsed

print("NEPHROLLM — FRESH BASELINE (REPARSED)")
print("=====================================")
print(f"Total questions:      {total_questions}")
print(f"Correct:              {correct}")
print(f"Incorrect:            {incorrect}")
print(f"Unparsed:             {unparsed}")
print(f"Accuracy (all):       {correct / total_questions:.2%}")
print(
    f"Accuracy (answered):  "
    f"{correct / (total_questions - unparsed):.2%}"
)

baseline_df.to_csv(
    "/kaggle/working/nephrollm_baseline_final_results.csv",
    index=False
)

NEPHROLLM — FRESH BASELINE (REPARSED)
Total questions:      636
Correct:              265
Incorrect:            328
Unparsed:             43
Accuracy (all):       41.67%
Accuracy (answered):  44.69%


# Final Inspection of Unparsed Responses

After re-scoring the complete baseline with the finalized parser, we perform one final inspection of the responses that remain unparsed.

This confirms that the parser is not systematically missing a valid answer format that should be counted.

For each remaining unparsed response, we retain:

- `BenchmarkID`
- expected answer
- available choices
- raw model response

The responses are kept as unparsed when the model does not provide a sufficiently clear option selection. We do not manually assign an answer during scoring.

This preserves a conservative and reproducible evaluation protocol.

**Expected outcome:** the remaining unparsed responses are confirmed as genuine non-answers or ambiguous outputs, supporting the use of the finalized baseline score.

In [18]:
remaining_unparsed = baseline_df[
    baseline_df["Predicted"].isna()
].copy()

print("Remaining unparsed:", len(remaining_unparsed))

display(
    remaining_unparsed[
        [
            "BenchmarkID",
            "Expected",
            "Choices",
            "ModelResponse"
        ]
    ]
)

Remaining unparsed: 43


,BenchmarkID,Expected,Choices,ModelResponse
24,NEPHRO-EVAL-0025,C,A. Intermittent hemodiaﬁltration\nB. Intermitt...,I'm happy to help you with your nephrology que...
64,NEPHRO-EVAL-0065,A,A. Regional citrate anticoagulation (RC...,I'm happy to help you with your nephrology que...
144,NEPHRO-EVAL-0145,D,A. Denosumab\nB. Increase calcium acetate\nC. ...,I'm happy to help you with your nephrology que...
155,NEPHRO-EVAL-0156,A,A. Cinacalcet therapy\nB. Calcitonin\nC. Estro...,I'm happy to help you with your nephrology que...
176,NEPHRO-EVAL-0177,C,1. Refer for cardiac catheterization\n2. Chest...,I'm happy to help you with your nephrology que...
180,NEPHRO-EVAL-0181,C,1. Ankylosing spondylitis\n2. Adynamic bone di...,I'm happy to help you with your nephrology que...
182,NEPHRO-EVAL-0183,D,1. Hypercalciuria\n2. Renal tubular acidosis\n...,I'm happy to help you with your nephrology que...
188,NEPHRO-EVAL-0189,C,1. A loss of function mutation in the calcium-...,I'm happy to help you with your nephrology que...
191,NEPHRO-EVAL-0192,A,1. Change calcium acetate to sevelamer carbona...,I can't provide a specific answer without know...
227,NEPHRO-EVAL-0228,D,A. Discontinue pravastatin now\nB. Discontinue...,I'm happy to help you with your nephrology que...


# 6.5 Analyze performance by subject

### Performance Across Nephrology Subjects

Overall accuracy provides a useful headline metric, but it does not show whether the model performs consistently across different areas of nephrology.

We therefore break the 636-question baseline down by the benchmark's **Subject** field.

For each subject, we examine:

- number of questions,
- number of correct answers,
- number of unparsed responses, and
- accuracy.

This helps identify areas where the base model performs relatively differently and provides a more informative starting point for evaluating the effect of nephrology-specific fine-tuning.

### Important interpretation note

Subject-level results should be interpreted alongside the number of questions in each category. Smaller subject groups can produce larger percentage changes from only a few questions.

The subject labels are retained exactly as provided by the benchmark rather than manually merging or renaming categories.

**Expected outcome:** a subject-level baseline performance table showing where the base model succeeds and where errors are concentrated.

In [19]:
subject_accuracy = (
    baseline_df
    .groupby("Subject")
    .agg(
        Questions=("BenchmarkID", "count"),
        Correct=("Correct", "sum"),
        Unparsed=("Predicted", lambda x: x.isna().sum())
    )
    .reset_index()
)

subject_accuracy["Incorrect"] = (
    subject_accuracy["Questions"]
    - subject_accuracy["Correct"]
    - subject_accuracy["Unparsed"]
)

subject_accuracy["Accuracy"] = (
    subject_accuracy["Correct"]
    / subject_accuracy["Questions"]
    * 100
)

subject_accuracy = subject_accuracy.sort_values(
    "Accuracy",
    ascending=True
)

display(subject_accuracy)


,Subject,Questions,Correct,Unparsed,Incorrect,Accuracy
15,Primary & Secondary Glomerular Diseases,23,6,7,10,26.086957
4,Disorders of Divalent Ions Renal Bone Disease ...,22,6,2,14,27.272727
5,"Disorders of Divalent Ions, Renal Bone Disease...",46,13,5,28,28.260870
14,Pregnancy and Kidney Disease,24,7,3,14,29.166667
17,Primary and Secondary Glomerular Diseases,48,14,5,29,29.166667
16,Primary Glomerular Disease,24,7,2,15,29.166667
8,Electrolytes and Acid-Base Disorders,22,7,2,13,31.818182
7,Electrolyte Acid-Base Disorders,24,8,4,12,33.333333
12,Infection Control & Prevention in Outpatient H...,24,9,2,13,37.500000
19,Transplantation,48,19,1,28,39.583333


# 6.6 Analyze incorrect answer selections


In [20]:
# Baseline error analysis — parseable incorrect answers only

baseline_errors = baseline_df[
    (baseline_df["Predicted"].notna()) &
    (baseline_df["Correct"] == False)
].copy()

print("BASELINE ERROR ANALYSIS")
print("=======================")
print(f"Total benchmark questions: {len(baseline_df)}")
print(f"Parseable incorrect answers: {len(baseline_errors)}")
print()

print("Most common predicted answers:")
display(
    baseline_errors["Predicted"]
    .value_counts()
    .rename_axis("Predicted")
    .reset_index(name="Count")
)

print("\nExpected answer distribution:")
display(
    baseline_errors["Expected"]
    .value_counts()
    .rename_axis("Expected")
    .reset_index(name="Count")
)

print("\nSample baseline errors:")
display(
    baseline_errors[
        [
            "BenchmarkID",
            "Subject",
            "Question",
            "Expected",
            "Predicted",
            "ModelResponse"
        ]
    ].head(20)
)

BASELINE ERROR ANALYSIS
Total benchmark questions: 636
Parseable incorrect answers: 328

Most common predicted answers:


,Predicted,Count
0,B,122
1,A,86
2,D,54
3,C,52
4,E,14



Expected answer distribution:


,Expected,Count
0,D,96
1,C,87
2,B,74
3,A,57
4,E,14



Sample baseline errors:


,BenchmarkID,Subject,Question,Expected,Predicted,ModelResponse
0,NEPHRO-EVAL-0001,Acute Kidney Injury & Critical Care Nephrology,Question 2824:\nThe transfusion of longer-term...,B,D,D
1,NEPHRO-EVAL-0002,Acute Kidney Injury & Critical Care Nephrology,"Based on data from the NUTRIREA-2 study, enter...",B,A,A
2,NEPHRO-EVAL-0003,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following is CORRECT about th...,B,A,A.
5,NEPHRO-EVAL-0006,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following individuals is MOST...,B,A,ANSWER: A
6,NEPHRO-EVAL-0007,Acute Kidney Injury & Critical Care Nephrology,"Based on the existing literature, implementing...",B,D,ANSWER: D
7,NEPHRO-EVAL-0008,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following outcomes has been d...,C,B,B
8,NEPHRO-EVAL-0009,Acute Kidney Injury & Critical Care Nephrology,Question 2833:\nImplementing a Kidney Disease ...,B,D,ANSWER: D
16,NEPHRO-EVAL-0017,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following should you tell the...,B,A,ANSWER: A
27,NEPHRO-EVAL-0028,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following is MOST likely to r...,B,C,ANSWER: 3
29,NEPHRO-EVAL-0030,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following interventions would...,D,C,ANSWER: 3


# 6.7 Analyze response formats


In [21]:
# Analyze baseline response formats

def classify_response_format(response):
    if response is None or str(response).strip() == "":
        return "empty"

    text = str(response).strip().upper()

    if re.search(r"\bANSWER\s*:\s*[A-E]\b", text):
        return "ANSWER: letter"

    if re.search(r"\bANSWER\s*:\s*[1-5]\b", text):
        return "ANSWER: number"

    if re.fullmatch(r"[A-E][\.\)]?", text):
        return "standalone letter"

    if re.fullmatch(r"[1-5][\.\)]?", text):
        return "standalone number"

    if text.startswith("I CANNOT") or text.startswith("I CAN'T"):
        return "refusal"

    if "MEDICAL ADVICE" in text:
        return "medical-advice refusal"

    return "other"


baseline_df["ResponseFormat"] = (
    baseline_df["ModelResponse"]
    .apply(classify_response_format)
)

format_summary = (
    baseline_df
    .groupby("ResponseFormat")
    .agg(
        Questions=("BenchmarkID", "count"),
        Correct=("Correct", "sum")
    )
    .reset_index()
)

format_summary["Accuracy"] = (
    format_summary["Correct"]
    / format_summary["Questions"]
    * 100
)

format_summary = format_summary.sort_values(
    "Questions",
    ascending=False
)

display(format_summary)

,ResponseFormat,Questions,Correct,Accuracy
0,ANSWER: letter,237,100,42.194093
1,ANSWER: number,181,81,44.751381
4,standalone letter,167,81,48.502994
2,other,41,2,4.878049
3,refusal,9,0,0.000000
5,standalone number,1,1,100.000000


# 6.8 Audit generated baseline artifacts

In [22]:
import os
import glob

print("Kaggle input datasets:")
print("======================")

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:20]:
        print(f"{indent}  {file}")

print("\nWorking directory CSV files:")
print("============================")

for path in sorted(glob.glob("/kaggle/working/*.csv")):
    print(path)

Kaggle input datasets:
input/
  datasets/
    shantanuss/
      nejm-ai-nephrology-benchmark/
        nephrollm_evaluation_manifest.csv
        nephrollm_primary_eval.csv
        nephrollm_heldout_eval.csv

Working directory CSV files:
/kaggle/working/nephrollm_baseline_final_batch_001_050.csv
/kaggle/working/nephrollm_baseline_final_batch_051_100.csv
/kaggle/working/nephrollm_baseline_final_batch_101_150.csv
/kaggle/working/nephrollm_baseline_final_batch_151_200.csv
/kaggle/working/nephrollm_baseline_final_batch_201_250.csv
/kaggle/working/nephrollm_baseline_final_batch_251_300.csv
/kaggle/working/nephrollm_baseline_final_batch_301_350.csv
/kaggle/working/nephrollm_baseline_final_batch_351_400.csv
/kaggle/working/nephrollm_baseline_final_batch_401_450.csv
/kaggle/working/nephrollm_baseline_final_batch_451_500.csv
/kaggle/working/nephrollm_baseline_final_batch_501_550.csv
/kaggle/working/nephrollm_baseline_final_batch_551_600.csv
/kaggle/working/nephrollm_baseline_final_batch_601_636.c

# 4.3 Load the benchmark manifest and held-out dataset

### Loading the Evaluation Manifest and Held-Out Split

We now load the benchmark manifest and the separate held-out evaluation dataset.

The **manifest** provides the official dataset structure and split assignments, while the **held-out dataset** contains the 160 questions reserved for the final generalization evaluation.

At this stage, we are only inspecting the data. No held-out questions are used for model training.

We verify:

- dataset dimensions,
- available columns,
- sample records, and
- the structure of the held-out evaluation data.

### Why this matters

The held-out split must remain separate from the QLoRA training corpus. Keeping it untouched allows us to evaluate whether the fine-tuned model can generalize to nephrology questions that were not used during training.

The held-out questions will be evaluated **after fine-tuning**, using the same answer-extraction and scoring methodology used for the primary evaluation.

**Expected outcome:** the manifest and held-out evaluation dataset load successfully, with the held-out split containing **160 questions**.

In [23]:
import pandas as pd

BASE = "/kaggle/input/datasets/shantanuss/nejm-ai-nephrology-benchmark"

manifest_path = f"{BASE}/nephrollm_evaluation_manifest.csv"
heldout_path = f"{BASE}/nephrollm_heldout_eval.csv"

manifest_df = pd.read_csv(manifest_path)
heldout_df = pd.read_csv(heldout_path)

print("MANIFEST")
print("========")
print("Shape:", manifest_df.shape)
print("Columns:", list(manifest_df.columns))
display(manifest_df.head(20))

print("\nHELD-OUT EVALUATION")
print("===================")
print("Shape:", heldout_df.shape)
print("Columns:", list(heldout_df.columns))
display(heldout_df.head(10))

MANIFEST
Shape: (796, 10)
Columns: ['BenchmarkID', 'ID', 'Subject', 'Context', 'Question', 'Choices', 'Answer', 'Solution', 'explanation', 'split']


,BenchmarkID,ID,Subject,Context,Question,Choices,Answer,Solution,explanation,split
0,NEPHRO-EVAL-0001,2824,Acute Kidney Injury & Critical Care Nephrology,NaN,Question 2824:\nThe transfusion of longer-term...,1. Increased risk of transfusion-related react...,B,temp nan,Answer B: No difference in mortality at 90 days,primary_eval
1,NEPHRO-EVAL-0002,2825,Acute Kidney Injury & Critical Care Nephrology,A 79-year-old man with stage G4 CKD and recent...,"Based on data from the NUTRIREA-2 study, enter...",A. Decreased incidence of systemic bacteremia\...,B,temp nan,Answer B: Increased incidence of gastrointest...,primary_eval
2,NEPHRO-EVAL-0003,2826,Acute Kidney Injury & Critical Care Nephrology,A 67-year-old woman with a history of stage G2...,Which ONE of the following is CORRECT about th...,1. She is at decreased risk for inpatient mort...,B,temp nan,Answer B: Her duration of RRT will be shorter,primary_eval
3,NEPHRO-EVAL-0004,2827,Acute Kidney Injury & Critical Care Nephrology,You are caring for a 71-year-old man with stag...,Which ONE of the following is associated with ...,A. A decreased risk of surgical site infection...,B,temp nan,Answer B: An increased risk of AKI requiring RRT,primary_eval
4,NEPHRO-EVAL-0005,2829,Acute Kidney Injury & Critical Care Nephrology,You are admitting a 65-year-old kidney trans-p...,Which ONE of the following is the MOST importa...,A. Administration of a bolus of 30 ml/kg 0.9% ...,D,temp nan,Answer D: Early adminstration of appropriate ...,primary_eval
5,NEPHRO-EVAL-0006,2830,Acute Kidney Injury & Critical Care Nephrology,You are meeting with your hospital’s chief inf...,Which ONE of the following individuals is MOST...,A. A 35-year-old woman with serum creatinine t...,B,temp nan,Answer B: A 76-year-old woman with a serum cr...,primary_eval
6,NEPHRO-EVAL-0007,2831,Acute Kidney Injury & Critical Care Nephrology,The chief quality ofﬁcer at your institution a...,"Based on the existing literature, implementing...",A. A reduction in nephrotoxin exposure and eve...,B,temp nan,"Answer B: A reduction in RRT events, inpatien...",primary_eval
7,NEPHRO-EVAL-0008,2832,Acute Kidney Injury & Critical Care Nephrology,"Several weeks after joining a new practice, yo...",Which ONE of the following outcomes has been d...,"A. Fewer cases of severe AKI, but no differenc...",C,temp nan,"Answer C: Fewer cases of severe AKI, a higher...",primary_eval
8,NEPHRO-EVAL-0009,2833,Acute Kidney Injury & Critical Care Nephrology,NaN,Question 2833:\nImplementing a Kidney Disease ...,A. Shorter length of mechanical ventilation\nB...,B,temp nan,Answer B: A decreased rate of stage 2 and 3 AKI,primary_eval
9,NEPHRO-EVAL-0010,2834,Acute Kidney Injury & Critical Care Nephrology,A 28-year-old woman with ESRD receives a de-ce...,This patient’s urine output in response to int...,A. Acute renal allograft rejection\nB. 1-year ...,C,temp nan,Answer C: Delayed graft function after renal ...,primary_eval



HELD-OUT EVALUATION
Shape: (160, 9)
Columns: ['BenchmarkID', 'ID', 'Subject', 'Context', 'Question', 'Choices', 'Answer', 'Solution', 'explanation']


,BenchmarkID,ID,Subject,Context,Question,Choices,Answer,Solution,explanation
0,NEPHRO-HOLDOUT-0001,2828,Acute Kidney Injury & Critical Care Nephrology,A 63-year-old kidney transplant recipient is s...,"In this setting, administration of bicarbonate...",A. Increased risk of mortality\nB. Increased r...,C,temp nan,Answer C: Decreased risk of progressive AKI r...
1,NEPHRO-HOLDOUT-0002,2848,Acute Kidney Injury & Critical Care Nephrology,A 65-year-old man with stage G3a CKD is seen i...,Which ONE of the following should you tell his...,A. Off-pump surgery in eligible patients is as...,A,temp nan,Answer A: Off-pump surgery in eligible patien...
2,NEPHRO-HOLDOUT-0003,5444,Acute Kidney Injury & Critical Care Nephrology,\nA 72-year-old man with hepatitis C–related c...,Which ONE of the following is the MOST likely ...,1. Abdominal compartment syndrome\n2. Acute tu...,D,temp nan,D. Hepatorenal syndrome
3,NEPHRO-HOLDOUT-0004,5448,Acute Kidney Injury & Critical Care Nephrology,\nYou are asked to see a 63-year-old man in co...,Which ONE of the following is MOST likely to p...,1. A fractional excretion of urea > 50%\n2. A ...,C,temp nan,C. Urine volume <200 ml over 2 hours after 1....
4,NEPHRO-HOLDOUT-0005,5455,Acute Kidney Injury & Critical Care Nephrology,NaN,Question 5455\nIn which ONE of the following c...,1. Coronary artery bypass surgery\n2. Open abd...,C,temp nan,C. Radiocontrast exposure
5,NEPHRO-HOLDOUT-0006,5457,Acute Kidney Injury & Critical Care Nephrology,\nYou are asked to see a 68-year-old man with ...,Which ONE of the following is the MOST appropr...,1. Give a ﬂuid bolus and reassess in 1 hour\n2...,A,temp nan,A. Give a fluid bolus and reassess in 1 hour
6,NEPHRO-HOLDOUT-0007,5463,Acute Kidney Injury & Critical Care Nephrology,\nA 34-year-old combat victim is ﬂown to a mil...,Which ONE of the following places this trauma ...,1. Chest injury\n2. Head injury\n3. Abdominal ...,C,temp nan,C. Abdominal injury
7,NEPHRO-HOLDOUT-0008,5467,Acute Kidney Injury & Critical Care Nephrology,\nA 47-year-old woman with advanced heart fail...,Which ONE of the following would you advise re...,1. She is likely to experience at least a tran...,A,temp nan,A. She is likely to experience at least a tran...
8,NEPHRO-HOLDOUT-0009,5471,Acute Kidney Injury & Critical Care Nephrology,\nAn 80-year-old man is hospitalized for AKI. ...,Which ONE of the following diagnostic tests is...,1. Serum creatine kinase\n2. Urine eosinophils...,C,temp nan,C. Kidney ultrasound
9,NEPHRO-HOLDOUT-0010,5510,Acute Kidney Injury & Critical Care Nephrology,"\nA 75-year-old woman with diabetes, hypertens...",What strategies can you implement to prevent AKI?,A. Check urine sodium and risk-stratify...,C,temp nan,C. Give 0.9% sodium chloride before and after ...


# 4.4 Inspect benchmark structure and data quality

We now perform a more detailed structural check of the benchmark manifest.

The inspection covers:

- overall dataset shape,
- column names,
- sample records,
- data types, and
- missing values.

This confirms that the manifest contains the fields required for evaluation and split management, and helps identify any data-quality issues before the evaluation workflow continues.

The manifest contains **796 records across 10 columns**.

We specifically check for missing values rather than silently filling them, because the evaluation data should be preserved in its original form.

**Expected outcome:** the manifest structure and data types are consistent with the expected benchmark schema, with any missing values explicitly reported for review.

# 4.5 Validate official evaluation splits

### Confirming the Benchmark Split Assignments

Before proceeding with model evaluation and fine-tuning, we verify the official split assignments provided by the benchmark manifest.

The benchmark contains two evaluation splits:

- **`primary_eval` — 636 questions**
- **`held_out` — 160 questions**

There is no training split in the benchmark manifest.

We also inspect the subject distribution within each split to understand how the evaluation data is organized.

### Why this validation matters

The split assignments determine how the benchmark will be used throughout the experiment:

- `primary_eval` is used for the base-model baseline and the later base-vs-fine-tuned comparison.
- `held_out` is reserved for the separate generalization evaluation.
- Neither split is permitted to become part of the QLoRA training corpus.

This check establishes the data boundary before any fine-tuning begins.

**Expected outcome:** the manifest should contain exactly **636 primary-evaluation questions and 160 held-out questions**, for a total of **796 evaluation questions**.

In [24]:
# Compact manifest inspection

print("Manifest shape:", manifest_df.shape)
print("Manifest columns:", list(manifest_df.columns))

print("\nFirst 10 rows:")
display(manifest_df.head(10))

print("\nData types:")
display(manifest_df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(
    manifest_df.isna().sum()
    .sort_values(ascending=False)
    .to_frame("Missing")
)

Manifest shape: (796, 10)
Manifest columns: ['BenchmarkID', 'ID', 'Subject', 'Context', 'Question', 'Choices', 'Answer', 'Solution', 'explanation', 'split']

First 10 rows:


,BenchmarkID,ID,Subject,Context,Question,Choices,Answer,Solution,explanation,split
0,NEPHRO-EVAL-0001,2824,Acute Kidney Injury & Critical Care Nephrology,NaN,Question 2824:\nThe transfusion of longer-term...,1. Increased risk of transfusion-related react...,B,temp nan,Answer B: No difference in mortality at 90 days,primary_eval
1,NEPHRO-EVAL-0002,2825,Acute Kidney Injury & Critical Care Nephrology,A 79-year-old man with stage G4 CKD and recent...,"Based on data from the NUTRIREA-2 study, enter...",A. Decreased incidence of systemic bacteremia\...,B,temp nan,Answer B: Increased incidence of gastrointest...,primary_eval
2,NEPHRO-EVAL-0003,2826,Acute Kidney Injury & Critical Care Nephrology,A 67-year-old woman with a history of stage G2...,Which ONE of the following is CORRECT about th...,1. She is at decreased risk for inpatient mort...,B,temp nan,Answer B: Her duration of RRT will be shorter,primary_eval
3,NEPHRO-EVAL-0004,2827,Acute Kidney Injury & Critical Care Nephrology,You are caring for a 71-year-old man with stag...,Which ONE of the following is associated with ...,A. A decreased risk of surgical site infection...,B,temp nan,Answer B: An increased risk of AKI requiring RRT,primary_eval
4,NEPHRO-EVAL-0005,2829,Acute Kidney Injury & Critical Care Nephrology,You are admitting a 65-year-old kidney trans-p...,Which ONE of the following is the MOST importa...,A. Administration of a bolus of 30 ml/kg 0.9% ...,D,temp nan,Answer D: Early adminstration of appropriate ...,primary_eval
5,NEPHRO-EVAL-0006,2830,Acute Kidney Injury & Critical Care Nephrology,You are meeting with your hospital’s chief inf...,Which ONE of the following individuals is MOST...,A. A 35-year-old woman with serum creatinine t...,B,temp nan,Answer B: A 76-year-old woman with a serum cr...,primary_eval
6,NEPHRO-EVAL-0007,2831,Acute Kidney Injury & Critical Care Nephrology,The chief quality ofﬁcer at your institution a...,"Based on the existing literature, implementing...",A. A reduction in nephrotoxin exposure and eve...,B,temp nan,"Answer B: A reduction in RRT events, inpatien...",primary_eval
7,NEPHRO-EVAL-0008,2832,Acute Kidney Injury & Critical Care Nephrology,"Several weeks after joining a new practice, yo...",Which ONE of the following outcomes has been d...,"A. Fewer cases of severe AKI, but no differenc...",C,temp nan,"Answer C: Fewer cases of severe AKI, a higher...",primary_eval
8,NEPHRO-EVAL-0009,2833,Acute Kidney Injury & Critical Care Nephrology,NaN,Question 2833:\nImplementing a Kidney Disease ...,A. Shorter length of mechanical ventilation\nB...,B,temp nan,Answer B: A decreased rate of stage 2 and 3 AKI,primary_eval
9,NEPHRO-EVAL-0010,2834,Acute Kidney Injury & Critical Care Nephrology,A 28-year-old woman with ESRD receives a de-ce...,This patient’s urine output in response to int...,A. Acute renal allograft rejection\nB. 1-year ...,C,temp nan,Answer C: Delayed graft function after renal ...,primary_eval



Data types:


,dtype
BenchmarkID,object
ID,int64
Subject,object
Context,object
Question,object
Choices,object
Answer,object
Solution,object
explanation,object
split,object



Missing values:


,Missing
Context,48
BenchmarkID,0
ID,0
Subject,0
Question,0
Choices,0
Answer,0
Solution,0
explanation,0
split,0


In [26]:
# Inspect the official benchmark split assignments

print("Official split distribution")
print("===========================")

split_counts = (
    manifest_df["split"]
    .value_counts()
    .rename_axis("Split")
    .reset_index(name="Questions")
)

display(split_counts)

print("\nSubject distribution by split")
print("=============================")

split_subject_counts = (
    manifest_df
    .groupby(["split", "Subject"])
    .size()
    .reset_index(name="Questions")
)

display(split_subject_counts)

Official split distribution


,Split,Questions
0,primary_eval,636
1,held_out,160



Subject distribution by split


,split,Subject,Questions
0,held_out,Acute Kidney Injury & Critical Care Nephrology,18
1,held_out,Chronic Kidney Disease,6
2,held_out,Chronic Kidney Disease and Progression,6
3,held_out,Disorder of Divalent Ions Renal Bone Disease a...,6
4,held_out,Disorders of Divalent Ions Renal Bone Disease ...,5
5,held_out,"Disorders of Divalent Ions, Renal Bone Disease...",11
6,held_out,ESRD & Dialysis,6
7,held_out,Electrolyte Acid-Base Disorders,6
8,held_out,Electrolytes and Acid-Base Disorders,6
9,held_out,End Stage Renal Disease & Dialysis,6


# 7. Prepare fine-tuning dataset


# 7.1 Define training-data requirements

The benchmark used in this project is reserved entirely for evaluation. It contains 636 primary-evaluation questions and 160 held-out questions, with no training split.

Therefore, the QLoRA training corpus must be a **separate nephrology educational dataset** that is not derived from the evaluation questions.

The training corpus should provide examples that are suitable for supervised instruction fine-tuning, with a clear relationship between an input and an expected educational response.

### Training-data requirements

The training corpus should:

- contain nephrology-focused educational content;
- be independent of the 636 primary evaluation questions;
- be independent of the 160 held-out questions;
- provide sufficiently clear question/answer or instruction/response examples;
- avoid relying on patient-specific medical advice;
- be convertible into the chat/instruction format expected by Llama 3.2 3B Instruct.

### Data-leakage rule

The evaluation benchmark is treated as **read-only evaluation data**.

No question, answer, solution, or explanation from the primary or held-out evaluation sets will be intentionally included as a QLoRA training example.

This separation is necessary so that the later comparison between the base model and fine-tuned model measures performance on unseen evaluation data rather than memorization.

**Expected outcome:** establish the requirements and data boundaries for the separate QLoRA training corpus before any training data is loaded or transformed.

# 7.2 Select an independent nephrology theory corpus


The evaluation benchmark does not contain a training split, so the QLoRA training data must come from a separate source.

For NephroLLM, we will use an **independent medical theory/textbook corpus** as the knowledge source for fine-tuning rather than training directly on multiple-choice questions.

### Why a theory corpus?

The goal of fine-tuning is to improve the model's underlying nephrology knowledge and its ability to provide educational explanations.

A theory-based corpus can expose the model to:

- nephrology concepts and terminology;
- disease mechanisms;
- renal physiology and pathophysiology;
- diagnostic principles;
- treatment concepts; and
- explanatory relationships between clinical concepts.

This provides a broader knowledge signal than training exclusively on MCQs.

### Training-data strategy

The selected theory corpus will be processed through the following pipeline:

```text
Independent medical theory corpus
            ↓
Identify nephrology-relevant content
            ↓
Clean and validate the text
            ↓
Remove duplicates and unsuitable content
            ↓
Check for overlap with the NephroLLM benchmark
            ↓
Convert selected content into
instruction-response examples
            ↓
Final QLoRA training dataset

# 7.3 Load and Inspect the Theory Corpus

We now load the independent medical theory corpus selected as the knowledge source for NephroLLM fine-tuning.

The purpose of this step is to understand the raw corpus before extracting nephrology-specific content.

We will inspect:

- the dataset structure and available splits;
- the available metadata;
- the source documents or textbooks;
- the text fields;
- the approximate size of the corpus; and
- whether the corpus provides enough information to reliably identify nephrology-related content.

At this stage, we will **not** filter, transform, or generate training examples.

Understanding the raw data first allows us to design the subsequent filtering and preprocessing steps based on the actual corpus structure rather than making assumptions about its fields or organization.

### Data boundary

The theory corpus is an independent training source.

The **636 primary-evaluation questions** and **160 held-out questions** remain outside this pipeline and will not be used to construct training examples.

**Expected outcome:** confirm that the theory corpus loads successfully and understand its schema and available metadata before extracting nephrology-relevant content.

In [8]:
from datasets import load_dataset

# Load the MedRAG medical textbook corpus
theory_ds = load_dataset("MedRAG/textbooks")

# Basic dataset structure
print("Dataset:")
print(theory_ds)

# Inspect available splits
print("\nSplits:")
print(theory_ds.keys())

# Inspect schema
print("\nFeatures:")
print(theory_ds["train"].features)

# Number of records
print("\nNumber of records:")
print(len(theory_ds["train"]))

# Inspect a sample record
print("\nSample record:")
print(theory_ds["train"][0])

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

chunk/Cell_Biology_Alberts.jsonl:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chunk/Gynecology_Novak.jsonl:   0%|          | 0.00/13.3M [00:00<?, ?B/s]

chunk/Histology_Ross.jsonl:   0%|          | 0.00/7.05M [00:00<?, ?B/s]

...

chunk/Immunology_Janeway.jsonl:   0%|          | 0.00/7.89M [00:00<?, ?B/s]

chunk/Psichiatry_DSM-5.jsonl:   0%|          | 0.00/6.73M [00:00<?, ?B/s]

chunk/Surgery_Schwartz.jsonl:   0%|          | 0.00/30.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/125847 [00:00<?, ? examples/s]

Dataset:
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'content', 'contents'],
        num_rows: 125847
    })
})

Splits:
dict_keys(['train'])

Features:
{'id': Value('string'), 'title': Value('string'), 'content': Value('string'), 'contents': Value('string')}

Number of records:
125847

Sample record:
{'id': 'Anatomy_Gray_0', 'title': 'Anatomy_Gray', 'content': 'What is anatomy? Anatomy includes those structures that can be seen grossly (without the aid of magnification) and microscopically (with the aid of magnification). Typically, when used by itself, the term anatomy tends to mean gross or macroscopic anatomy—that is, the study of structures that can be seen without using a microscopic. Microscopic anatomy, also called histology, is the study of cells and tissues using a microscope. Anatomy forms the basis for the practice of medicine. Anatomy leads the physician toward an understanding of a patient’s disease, whether he or she is carrying out a physical ex

# 7.4 Identify Nephrology-Relevant Content

The theory corpus contains medical content across multiple subjects and clinical domains. We now identify the portions that are relevant to nephrology and renal medicine.

The goal is to construct a focused nephrology knowledge corpus rather than fine-tuning on unrelated medical material.

### Selection approach

We will use the metadata and text available in the corpus to identify relevant content.

Potential nephrology-related concepts include:

- kidney and renal physiology;
- acute kidney injury;
- chronic kidney disease;
- glomerular diseases;
- electrolyte and acid-base disorders;
- hypertension and kidney disease;
- dialysis and end-stage kidney disease;
- renal transplantation;
- nephrolithiasis;
- renal pathology; and
- other kidney-related conditions and mechanisms.

The exact filtering criteria will be determined from the actual structure and metadata of the loaded corpus.

### Important consideration

Keyword matching alone can produce both false positives and false negatives.

For example, a general medical passage may mention the kidney without being primarily about nephrology, while a relevant nephrology concept may not contain an obvious keyword.

Therefore, the initial filter will be treated as a **candidate-selection step**. The resulting records will be inspected before they become part of the final training corpus.

**Expected outcome:** identify a reproducible method for extracting candidate nephrology-related passages from the independent theory corpus.

In [9]:
import pandas as pd
import re

# Convert the training split to a DataFrame
theory_df = theory_ds["train"].to_pandas()

# Textbooks most likely to contain substantial nephrology / renal content
relevant_books = [
    "InternalMed_Harrison",
    "Physiology_Levy",
    "Pathology_Robbins",
    "Pathoma_Husain",
    "Anatomy_Gray",
    "Pharmacology_Katzung",
]

# Nephrology / renal keywords used for candidate selection
nephro_keywords = [
    "nephrology",
    "nephrolog",
    "renal",
    "kidney",
    "glomerul",
    "nephron",
    "glomerulonephritis",
    "nephritis",
    "nephrotic",
    "glomerulosclerosis",
    "acute kidney injury",
    "acute renal failure",
    "chronic kidney disease",
    "chronic renal failure",
    "dialysis",
    "hemodialysis",
    "peritoneal dialysis",
    "renal replacement",
    "kidney transplant",
    "renal transplant",
    "renal transplantation",
    "electrolyte",
    "acid-base",
    "hyperkalemia",
    "hypokalemia",
    "hyponatremia",
    "hypernatremia",
    "renal stone",
    "kidney stone",
    "nephrolithiasis",
    "polycystic kidney",
    "renal artery",
    "renal tubular",
    "tubulointerstitial",
]

# Restrict to selected medical textbooks
candidate_df = theory_df[
    theory_df["title"].isin(relevant_books)
].copy()

# Normalize text for case-insensitive matching
candidate_df["content_lower"] = (
    candidate_df["content"]
    .fillna("")
    .str.lower()
)

# Escape keywords before constructing the regex
keyword_pattern = "|".join(
    map(re.escape, nephro_keywords)
)

# Identify candidate nephrology chunks
candidate_df = candidate_df[
    candidate_df["content_lower"].str.contains(
        keyword_pattern,
        regex=True,
        na=False
    )
].copy()

# Remove temporary column
candidate_df.drop(columns=["content_lower"], inplace=True)

print("Candidate nephrology chunks:", len(candidate_df))

print("\nCandidate chunks by textbook:")
print(
    candidate_df["title"]
    .value_counts()
    .sort_index()
)

Candidate nephrology chunks: 6242

Candidate chunks by textbook:
title
Anatomy_Gray             136
InternalMed_Harrison    3694
Pathology_Robbins        600
Pathoma_Husain            95
Pharmacology_Katzung    1024
Physiology_Levy          693
Name: count, dtype: int64


Inspect Candidate Nephrology Content

In [10]:
# Inspect representative candidate chunks from each source textbook

sample_per_book = 3

for book in relevant_books:
    book_samples = candidate_df[
        candidate_df["title"] == book
    ].sample(
        n=min(sample_per_book, len(candidate_df[candidate_df["title"] == book])),
        random_state=42
    )

    print("\n" + "=" * 100)
    print(f"SOURCE: {book}")
    print("=" * 100)

    for i, row in book_samples.iterrows():
        print(f"\nID: {row['id']}")
        print(f"TEXT:\n{row['content'][:1500]}")
        print("-" * 100)


SOURCE: InternalMed_Harrison

ID: InternalMed_Harrison_3588
TEXT:
Immediate antagonism of the cardiac effects of hyperkalemia. Intravenous calcium serves to protect the heart, whereas other measures are taken to correct hyperkalemia. Calcium raises the action potential threshold and reduces excitability, without changing the resting membrane potential. By restoring the difference between resting and threshold potentials, calcium reverses the depolarization blockade due to hyperkalemia. The recommended dose is 10 mL of 10% calcium gluconate (3–4 mL of calcium chloride), infused intravenously over 2–3 min with cardiac monitoring. The effect of the infusion starts in 1–3 min and lasts 30–60 min; the dose should be repeated if there is no change in ECG findings or if they recur after initial improvement. Hypercalcemia potentiates the cardiac toxicity of digoxin; hence, intravenous calcium should be used with extreme caution in patients taking this medication; if judged necessary, 10 mL of

In [11]:
# Stronger nephrology-specific terms for second-stage filtering

strong_nephro_keywords = [
    "glomerul",
    "nephron",
    "nephritis",
    "nephrotic",
    "nephropathy",
    "nephrolithiasis",
    "glomerulosclerosis",
    "glomerulonephritis",
    "acute kidney injury",
    "acute renal failure",
    "chronic kidney disease",
    "chronic renal failure",
    "end-stage renal",
    "end stage renal",
    "renal replacement therapy",
    "renal replacement",
    "hemodialysis",
    "haemodialysis",
    "peritoneal dialysis",
    "dialysis",
    "kidney transplant",
    "renal transplant",
    "renal transplantation",
    "renal tubular",
    "tubular acidosis",
    "tubulointerstitial",
    "renal artery",
    "renal vein",
    "renal blood flow",
    "glomerular filtration",
    "glomerular filtration rate",
    "glomerular capillary",
    "glomerular disease",
    "renal stone",
    "kidney stone",
    "renal calcul",
    "polycystic kidney",
    "renal failure",
    "renal insufficiency",
    "renal disease",
    "renal physiology",
    "renal pathology",
]

strong_pattern = "|".join(
    map(re.escape, strong_nephro_keywords)
)

# Apply the stronger filter to the existing candidate pool
candidate_df["content_lower"] = (
    candidate_df["content"]
    .fillna("")
    .str.lower()
)

nephro_df = candidate_df[
    candidate_df["content_lower"].str.contains(
        strong_pattern,
        regex=True,
        na=False
    )
].copy()

nephro_df.drop(columns=["content_lower"], inplace=True)

print("Initial candidate chunks:", len(candidate_df))
print("Stronger nephrology candidates:", len(nephro_df))

print("\nCandidates by textbook:")
print(
    nephro_df["title"]
    .value_counts()
    .sort_index()
)

Initial candidate chunks: 6242
Stronger nephrology candidates: 3148

Candidates by textbook:
title
Anatomy_Gray              42
InternalMed_Harrison    2023
Pathology_Robbins        310
Pathoma_Husain            58
Pharmacology_Katzung     431
Physiology_Levy          284
Name: count, dtype: int64


Validate the Refined Nephrology Candidates

In [12]:
# Inspect representative examples from the refined nephrology candidate set

sample_per_book = 5

for book in relevant_books:
    book_df = nephro_df[nephro_df["title"] == book]

    if len(book_df) == 0:
        continue

    samples = book_df.sample(
        n=min(sample_per_book, len(book_df)),
        random_state=42
    )

    print("\n" + "=" * 100)
    print(f"SOURCE: {book} | CANDIDATES: {len(book_df)}")
    print("=" * 100)

    for _, row in samples.iterrows():
        print(f"\nID: {row['id']}")
        print(f"TEXT:\n{row['content'][:1800]}")
        print("-" * 100)


SOURCE: InternalMed_Harrison | CANDIDATES: 2023

ID: InternalMed_Harrison_16276
TEXT:
The diminished effectiveness of eflornithine against T. b. rhodesiense appears to be due to the parasite’s ability to replace the inhibited enzyme more rapidly than T. b. gambiense. Eflornithine is less toxic but more costly than conventional therapy. It can be administered IV or PO. The dose should be reduced in renal failure. Eflornithine readily crosses the blood-brain barrier; CSF levels are highest in persons with the most severe central nervous system (CNS) involvement.
----------------------------------------------------------------------------------------------------

ID: InternalMed_Harrison_21539
TEXT:
The urinalysis and urine sediment examination are invaluable tools, but they require clinical correlation because of generally limited sensitivity and specificity (Fig. 334-6) (Chap. 62e). In the absence of preexisting proteinuria from CKD, AKI from ischemia or nephrotoxins leads to mild prot

# 7.5 Clean and Prepare the Training Text

After identifying candidate nephrology content, we now prepare the selected text for use in supervised fine-tuning.

The objective is to retain useful medical knowledge while removing records that are unsuitable or likely to introduce noise into the training process.

### Cleaning steps

The preprocessing stage will examine the selected records for:

- missing or empty text;
- duplicate passages;
- extremely short or excessively long passages;
- malformed records;
- unnecessary formatting or metadata; and
- passages that do not contain meaningful educational content.

We will preserve the original educational meaning of the source material while applying only transformations necessary for training-data preparation.

### Why cleaning matters

Fine-tuning quality depends not only on the quantity of training data but also on the consistency and usefulness of the examples.

Removing duplicates and unusable records helps prevent the model from receiving repeated or low-information training examples.

At the same time, we avoid aggressive preprocessing that could remove medically meaningful terminology or context.

**Expected outcome:** a cleaned set of nephrology-focused educational passages suitable for conversion into instruction-response training examples.

In [13]:
# Basic cleaning and quality checks for the nephrology candidate corpus

import re
import pandas as pd

clean_df = nephro_df.copy()

# Normalize whitespace while preserving the original medical content
clean_df["content_clean"] = (
    clean_df["content"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Basic text statistics
clean_df["char_count"] = clean_df["content_clean"].str.len()
clean_df["word_count"] = clean_df["content_clean"].str.split().str.len()

print("Initial nephrology candidates:", len(clean_df))

print("\nMissing / empty content:")
print(
    (clean_df["content_clean"].str.len() == 0).sum()
)

print("\nText length statistics:")
print(
    clean_df["word_count"].describe()
)

print("\nVery short passages (<50 words):")
print(
    (clean_df["word_count"] < 50).sum()
)

print("\nVery long passages (>1500 words):")
print(
    (clean_df["word_count"] > 1500).sum()
)

print("\nDuplicate passages:")
print(
    clean_df["content_clean"].duplicated().sum()
)

Initial nephrology candidates: 3148

Missing / empty content:
0

Text length statistics:
count    3148.000000
mean      110.871982
std        35.102878
min         1.000000
25%        90.000000
50%       121.000000
75%       138.000000
max       178.000000
Name: word_count, dtype: float64

Very short passages (<50 words):
236

Very long passages (>1500 words):
0

Duplicate passages:
0


inspect short passages

In [14]:
# Inspect short nephrology passages before deciding whether to exclude them

short_df = clean_df[
    clean_df["word_count"] < 50
].copy()

print("Short passages:", len(short_df))

print("\nShort passages by textbook:")
print(
    short_df["title"]
    .value_counts()
    .sort_index()
)

print("\nRepresentative short passages:")

for book in relevant_books:
    book_samples = short_df[
        short_df["title"] == book
    ]

    if len(book_samples) == 0:
        continue

    samples = book_samples.sample(
        n=min(5, len(book_samples)),
        random_state=42
    )

    print("\n" + "=" * 100)
    print(f"SOURCE: {book}")
    print("=" * 100)

    for _, row in samples.iterrows():
        print(f"\nID: {row['id']}")
        print(f"WORDS: {row['word_count']}")
        print(f"TEXT: {row['content_clean']}")
        print("-" * 100)

Short passages: 236

Short passages by textbook:
title
InternalMed_Harrison    126
Pathology_Robbins        78
Pharmacology_Katzung     16
Physiology_Levy          16
Name: count, dtype: int64

Representative short passages:

SOURCE: InternalMed_Harrison

ID: InternalMed_Harrison_21867
WORDS: 44
TEXT: The subject of lupus nephritis is presented under acute nephritic syndromes because of the aggressive and important proliferative lesions seen in class III–V renal disease. Class III describes focal lesions Note: Revised in 2004 by the International Society of Nephrology-Renal Pathology Society Study Group.
----------------------------------------------------------------------------------------------------

ID: InternalMed_Harrison_10627
WORDS: 43
TEXT: supported these laboratory results, and antibiotics may increase by twentyfold the risk of hemolytic-uremic syndrome and renal failure during enterohemorrhagic E. coli infection. A clinical clue in the diagnosis of the latter infection is 

remove obvious structural noise and duplicates

We'll first remove only clearly unusable records:

empty text;
extremely tiny fragments (<10 words);
obvious figure/table fragments;
duplicate text.

We will keep passages of 10+ words for now, even if they're short, because some contain dense nephrology knowledge.

In [15]:
# Apply conservative cleaning rules to the nephrology candidate corpus

before_count = len(clean_df)

# Remove empty passages
clean_df = clean_df[
    clean_df["content_clean"].str.len() > 0
].copy()

# Remove extremely short fragments
# We keep passages with >=10 words because some short passages
# contain dense nephrology concepts.
clean_df = clean_df[
    clean_df["word_count"] >= 10
].copy()

# Remove exact duplicate passages
clean_df = clean_df.drop_duplicates(
    subset=["content_clean"]
).copy()

# Reset index after filtering
clean_df.reset_index(drop=True, inplace=True)

after_count = len(clean_df)

print("Candidate passages before cleaning:", before_count)
print("Candidate passages after cleaning:", after_count)
print("Removed passages:", before_count - after_count)

print("\nRemaining passages by textbook:")
print(
    clean_df["title"]
    .value_counts()
    .sort_index()
)

print("\nRemaining word-count statistics:")
print(
    clean_df["word_count"].describe()
)

Candidate passages before cleaning: 3148
Candidate passages after cleaning: 3113
Removed passages: 35

Remaining passages by textbook:
title
Anatomy_Gray              42
InternalMed_Harrison    2021
Pathology_Robbins        282
Pathoma_Husain            58
Pharmacology_Katzung     431
Physiology_Levy          279
Name: count, dtype: int64

Remaining word-count statistics:
count    3113.000000
mean      112.058786
std        33.455656
min        10.000000
25%        91.000000
50%       121.000000
75%       138.000000
max       178.000000
Name: word_count, dtype: float64


# 7.6 Convert Theory into Instruction-Response Examples

The cleaned nephrology passages are still raw educational text. We now transform them into structured **instruction-response examples** suitable for supervised fine-tuning.

The objective is to teach Llama 3.2 3B Instruct to use nephrology knowledge when responding to educational prompts rather than simply memorizing isolated text passages.

### Example structure

Each training example will follow a simple instruction-response format:

```text
Instruction:
Explain the role of the glomerulus in renal filtration.

Response:
The glomerulus is a network of specialized capillaries...

**Why are we converting to Instruction response format ?**

Because Llama 3.2 3B Instruct is an instruction-tuned model, and our goal is not simply to make it memorize textbook passages. We want QLoRA to improve its ability to answer nephrology questions and explain nephrology concepts.

There are two different training approaches:
| Approach                         | What the model sees                  | What it teaches                                                 |
| -------------------------------- | ------------------------------------ | --------------------------------------------------------------- |
| Raw text / continued pretraining | `"The glomerulus filters plasma..."` | More exposure to nephrology language/knowledge                  |
| Instruction-response SFT         | **Question → Answer**                | How to use nephrology knowledge to answer educational questions |


**Identify Passages Suitable for Instruction Generation**

Before generating instruction-response pairs, let's first identify passages that contain enough coherent nephrology information to support an educational question and answer.

For this step, we'll use simple heuristics, not an LLM. This keeps the selection reproducible and avoids introducing synthetic judgments too early.

In [16]:
# Identify cleaned passages that are suitable candidates
# for instruction-response generation.

instruction_candidates = clean_df.copy()

# Basic suitability signals
instruction_candidates["has_explanatory_language"] = (
    instruction_candidates["content_clean"]
    .str.contains(
        r"\b(is|are|refers to|defined|causes|results in|leads to|"
        r"associated with|characterized by|regulates|inhibits|"
        r"stimulates|reabsorbs|excretes|increases|decreases)\b",
        case=False,
        regex=True,
        na=False
    )
)

instruction_candidates["has_nephro_term"] = (
    instruction_candidates["content_clean"]
    .str.contains(
        r"\b(kidney|renal|nephron|nephritic|nephrotic|"
        r"glomerul|dialysis|glomerular|tubular|"
        r"ureter|urine|creatinine|GFR|"
        r"aldosterone|renin|electrolyte|acid-base)\b",
        case=False,
        regex=True,
        na=False
    )
)

# Keep passages that contain enough text and at least
# one explanatory or nephrology-specific signal.
instruction_candidates = instruction_candidates[
    (instruction_candidates["word_count"] >= 20) &
    (
        instruction_candidates["has_explanatory_language"] |
        instruction_candidates["has_nephro_term"]
    )
].copy()

print("Cleaned passages:", len(clean_df))
print("Instruction-generation candidates:", len(instruction_candidates))
print(
    "Candidate retention:",
    f"{len(instruction_candidates) / len(clean_df) * 100:.1f}%"
)

print("\nCandidates by textbook:")
print(
    instruction_candidates["title"]
    .value_counts()
    .sort_index()
)

# Remove helper columns before continuing
instruction_candidates.drop(
    columns=[
        "has_explanatory_language",
        "has_nephro_term"
    ],
    inplace=True
)

Cleaned passages: 3113
Instruction-generation candidates: 3024
Candidate retention: 97.1%

Candidates by textbook:
title
Anatomy_Gray              41
InternalMed_Harrison    1981
Pathology_Robbins        246
Pathoma_Husain            58
Pharmacology_Katzung     427
Physiology_Levy          271
Name: count, dtype: int64


/tmp/ipykernel_58/2330636703.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(
/tmp/ipykernel_58/2330636703.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(



3,024 of 3,113 passages (97.1%) pass the basic suitability heuristic. That tells us the heuristic is intentionally permissive—which is good here. We don't want to discard potentially useful medical content before we have actually transformed it.

**Define the Instruction-Response Format**

The selected theory passages now need to be transformed into supervised fine-tuning examples.

Rather than generating arbitrary questions, we will use a small set of controlled educational instruction types.

### Instruction types

Depending on the information available in a passage, an example may ask the model to:

- **Explain a concept** — explain a nephrology concept or mechanism.
- **Describe a process** — describe how a renal physiological or pathological process works.
- **Explain a clinical concept** — explain a disease, syndrome, finding, or complication.
- **Explain a relationship** — explain how two nephrology concepts are related.
- **Compare concepts** — compare two concepts when the source passage explicitly supports the comparison.

Not every passage needs to support every instruction type.

### Example

For a passage describing glomerular filtration:

```text
Instruction:
Explain the role of the glomerulus in renal filtration.

Response:
The glomerulus is a specialized network of capillaries that filters
plasma to form the initial tubular fluid...

**Create a Pilot Instruction-Response Dataset**

In [17]:
# Create a small pilot dataset from the selected nephrology passages.
# The pilot will be inspected before generating the full training set.

pilot_size = min(50, len(instruction_candidates))

pilot_df = instruction_candidates.sample(
    n=pilot_size,
    random_state=42
).copy()

# Define the instruction-response dataset schema.
# The response will be generated in the next step.
pilot_df["instruction"] = ""
pilot_df["response"] = ""

# Keep source information for traceability.
pilot_training_df = pilot_df[
    [
        "id",
        "title",
        "content_clean",
        "instruction",
        "response"
    ]
].copy()

print("Pilot passages:", len(pilot_training_df))

print("\nDataset columns:")
print(pilot_training_df.columns.tolist())

print("\nSample source passages:")
for _, row in pilot_training_df.head(5).iterrows():
    print("\n" + "=" * 80)
    print(f"Source: {row['title']}")
    print(f"ID: {row['id']}")
    print(f"Text: {row['content_clean']}")

Pilot passages: 50

Dataset columns:
['id', 'title', 'content_clean', 'instruction', 'response']

Sample source passages:

Source: Physiology_Levy
ID: Physiology_Levy_4128
Text: Actions of Cortisol on the Kidney Cortisol inhibits the secretion and action of antidiuretic hormone (ADH), and thus it is an ADH antagonist. In the absence of cortisol, the action of ADH is potentiated, which makes it difficult to increase free water clearance in response to a water load and increases the likelihood of water intoxication. Although cortisol binds to the mineralocorticoid receptor with high affinity, this action is normally blocked by inactivation of cortisol to cortisone by the enzyme 11βHSD2. However, the mineralocorticoid activity (i.e., renal Na+ and H2O retention, K+ and H+ excretion) of cortisol depends on the relative amount of cortisol (or synthetic glucocorticoids) and the activity of 11β-HSD2. Certain agents (e.g., compounds in black licorice) inhibit 11βHSD2 and thereby increase the m

The pilot reveals exactly why we did this check. The passages are not all equally suitable for SFT generation.

For example:

The cortisol/kidney passage is coherent and can support several educational questions.
The dRTA passage is also rich enough for an educational example.
The BUN/creatinine passage is strong.
But the thyroid passage contains an incidental renal reference.
The Sjögren's passage is essentially a table/summary fragment, not a good standalone teaching passage.

So before generating synthetic instructions, **we should add one more quality-control layer.**

In [19]:
# Identify passages that are likely to be structurally noisy,
# such as table fragments, figure labels, or broken text.

quality_df = instruction_candidates.copy()

text = quality_df["content_clean"]

# Signals of potentially noisy textbook extraction
quality_df["has_table_signal"] = text.str.contains(
    r"\b(Table|TABLE|Fig\.|Figure|FIGURE)\b",
    regex=True,
    na=False
)

quality_df["has_fragment_signal"] = (
    text.str.match(r"^[a-z]", na=False) |
    text.str.contains(r"\.{2,}|:{2,}", regex=True, na=False)
)

quality_df["has_many_short_tokens"] = (
    text.str.split().apply(
        lambda words: (
            len(words) > 0 and
            sum(len(w) <= 2 for w in words) / len(words) > 0.25
        )
    )
)

# Count passages with each signal
print("Instruction candidates:", len(quality_df))

print("\nPotential table/figure passages:")
print(quality_df["has_table_signal"].sum())

print("\nPotential fragmented passages:")
print(quality_df["has_fragment_signal"].sum())

print("\nPassages with many very short tokens:")
print(quality_df["has_many_short_tokens"].sum())

# Show examples flagged by the structural checks
flagged_df = quality_df[
    quality_df["has_table_signal"] |
    quality_df["has_fragment_signal"] |
    quality_df["has_many_short_tokens"]
].copy()

print("\nTotal flagged for review:", len(flagged_df))

for _, row in flagged_df.head(15).iterrows():
    print("\n" + "=" * 100)
    print(f"SOURCE: {row['title']}")
    print(f"ID: {row['id']}")
    print(f"TEXT:\n{row['content_clean']}")

Instruction candidates: 3024

Potential table/figure passages:
304

Potential fragmented passages:
592

Passages with many very short tokens:
168

Total flagged for review: 996

SOURCE: Anatomy_Gray
ID: Anatomy_Gray_777
TEXT:
Anterior to the abdominal aorta, as it descends, are the pancreas and splenic vein, the left renal vein, and the inferior part of the duodenum. Several left lumbar veins cross it posteriorly as they pass to the inferior vena cava. On its right side are the cisterna chyli, thoracic duct, azygos vein, right crus of the diaphragm, and the inferior vena cava. On its left side is the left crus of the diaphragm. Branches of the abdominal aorta (Table 4.3) can be classified as: visceral branches supplying organs, posterior branches supplying the diaphragm or body wall, or terminal branches. The visceral branches are either unpaired or paired vessels. The three unpaired visceral branches that arise from the anterior surface of the abdominal aorta (Fig. 4.164) are: the cel

/tmp/ipykernel_58/1116769080.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  quality_df["has_table_signal"] = text.str.contains(


This confirms the structural filter is too broad to use as an automatic exclusion rule.

We have 996 flagged passages, but several are actually useful. For example, a passage can contain a legitimate renal concept while also mentioning a figure/table. Conversely, some flagged passages are clearly incidental—for example, renal disease appearing only as one risk factor or comorbidity.

The purpose of the generation prompt should therefore be simply:

**Take an already-selected nephrology passage and create one instruction-response example grounded in that passage.**

In [41]:
def create_teacher_training_example_prompt(passage):
    return f"""Create ONE high-quality educational Q&A from this
nephrology-focused textbook passage.

SOURCE PASSAGE:
{passage}

IMPORTANT SELECTION RULE:

The Q&A must teach the CENTRAL renal/nephrology concept of the
passage.

Do NOT create a Q&A merely because the passage mentions:
- renal impairment,
- kidney function,
- renal drug clearance,
- dose adjustment in renal disease,
- a kidney-related complication of another disease, or
- an electrolyte abnormality in a primarily non-renal context.

Prefer passages whose central teaching point concerns:
- renal physiology,
- glomerular or tubular function,
- kidney disease,
- renal pathology,
- electrolyte or acid-base physiology when the renal mechanism
  is central,
- acute or chronic kidney disease,
- dialysis,
- transplantation,
- renal diagnostic findings, or
- renal treatment principles.

If the central teaching point is primarily another medical specialty,
output exactly:

SKIP

If the passage supports a strong nephrology teaching example:

1. Write ONE clear educational question.
2. Answer it using ONLY information explicitly stated in the passage.
3. Do not add outside medical knowledge.
4. Do not infer facts that are not stated.
5. Keep the answer concise.
6. Do not provide patient-specific medical advice.

Output ONLY:

INSTRUCTION: <actual educational question>
RESPONSE: <actual answer>

OR:

SKIP
"""

**Generate 10 Pilot Instruction-Response Examples**

In [44]:
# Generate a small pilot of instruction-response examples

import torch
import pandas as pd

pilot_results = []

# Use 10 passages from our existing 50-passage pilot
generation_pilot = pilot_training_df.head(10)

model.eval()

for _, row in generation_pilot.iterrows():

    prompt = create_training_example_prompt(
        row["content_clean"]
    )

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    pilot_results.append({
        "source_id": row["id"],
        "source_title": row["title"],
        "source_text": row["content_clean"],
        "generated_example": generated_text
    })

# Store the results without modifying the original dataset
generated_pilot_df = pd.DataFrame(pilot_results)

print("Generated examples:", len(generated_pilot_df))

for _, row in generated_pilot_df.iterrows():
    print("\n" + "=" * 90)
    print("SOURCE ID:", row["source_id"])
    print("SOURCE:", row["source_title"])
    print("\nGENERATED EXAMPLE:")
    print(row["generated_example"])

Generated examples: 10

SOURCE ID: Physiology_Levy_4128
SOURCE: Physiology_Levy

GENERATED EXAMPLE:
INSTRUCTION: What is the effect of cortisol on the action of antidiuretic hormone (ADH) in the kidney?
RESPONSE: Cortisol inhibits the secretion and action of ADH, making it an ADH antagonist.

SOURCE ID: InternalMed_Harrison_3651
SOURCE: InternalMed_Harrison

GENERATED EXAMPLE:
INSTRUCTION: What is the primary renal tubular abnormality in a patient with classic distal renal tubular acidosis (dRTA)?

RESPONSE: A renal tubular abnormality that causes abnormally low excretion of ammonium in the face of systemic acidosis.

SOURCE ID: InternalMed_Harrison_26834
SOURCE: InternalMed_Harrison

GENERATED EXAMPLE:
INSTRUCTION: What is the effect of renal disease on T3 levels in patients with HIV infection?
RESPONSE: T3 levels fall with progression to AIDS.

SOURCE ID: InternalMed_Harrison_25457
SOURCE: InternalMed_Harrison

GENERATED EXAMPLE:
INSTRUCTION: What is the standard treatment for lympho

The problem we're seeing isn't that the passages are bad. It's that Llama 3.2 3B is being asked to perform a fairly demanding data-curation task: understand the passage, determine the central concept, decide whether it is nephrology-specific, and then generate a grounded Q&A. The pilot shows it sometimes fails at that.

Better architecture

Instead of:

Textbook passage
      ↓
Llama 3.2 3B
      ↓
Q&A


use:

Textbook passage
      ↓
Stronger instruction model Qwen2.5 7B Instruct
      ↓
High-quality Q&A dataset
      ↓
Llama 3.2 3B + QLoRA
      ↓
NephroLLM

**7.6 Load the Teacher Model for Synthetic Dataset Construction**

The initial pilot showed that Llama 3.2 3B can generate correctly formatted Q&A examples, but it sometimes focuses on a secondary or incidental medical concept rather than the main nephrology concept in the source passage.

To improve the quality of the supervised fine-tuning dataset, we introduce a separate **teacher model** for synthetic instruction-response generation.

We will use **Qwen2.5-7B-Instruct** as the teacher model and retain **Llama 3.2 3B-Instruct** as the target model that will ultimately be fine-tuned.

The roles are therefore separated:

```text
Independent medical textbook corpus
            ↓
Qwen2.5-7B-Instruct
     Teacher / data generator
            ↓
High-quality nephrology Q&A
            ↓
Llama 3.2 3B-Instruct
     Target model + QLoRA

In [20]:

# Load Qwen2.5-7B-Instruct as the teacher model
# for synthetic nephrology training-data generation.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

teacher_model_name = "Qwen/Qwen2.5-7B-Instruct"

# 4-bit quantization for the Kaggle T4 GPU
teacher_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# Load tokenizer
teacher_tokenizer = AutoTokenizer.from_pretrained(
    teacher_model_name,
    token=hf_token,
)

# Load quantized teacher model
teacher_model = AutoModelForCausalLM.from_pretrained(
    teacher_model_name,
    quantization_config=teacher_bnb_config,
    device_map="auto",
    token=hf_token,
)

teacher_model.eval()

print("Teacher model loaded successfully.")
print("Model:", teacher_model_name)
print("Device:", teacher_model.device)
print("Dtype:", teacher_model.dtype)
print("4-bit quantization: enabled")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Teacher model loaded successfully.
Model: Qwen/Qwen2.5-7B-Instruct
Device: cuda:0
Dtype: torch.bfloat16
4-bit quantization: enabled


**7.6.1 Verify Teacher-Model Inference**

Before using the teacher model to construct the synthetic training dataset, we verify that Qwen2.5-7B-Instruct can generate responses correctly using its native chat template.

This is a technical sanity check only. The teacher model is not yet being used to generate the training corpus.

We use a simple nephrology question to confirm:

- the tokenizer and chat template work correctly;
- the quantized model can generate on the available GPU;
- the output is coherent; and
- the teacher model is ready for the dataset-generation pilot.

**Expected outcome:** a coherent educational nephrology response from Qwen2.5-7B-Instruct.

In [21]:
# Verify Qwen teacher-model inference

def ask_teacher(question, max_new_tokens=128):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an educational assistant specializing in nephrology. "
                "Provide clear, factual, educational explanations. "
                "Do not provide diagnosis or personalized medical advice."
            ),
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    prompt = teacher_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = teacher_tokenizer(
        prompt,
        return_tensors="pt",
    ).to(teacher_model.device)

    with torch.no_grad():
        outputs = teacher_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=teacher_tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = teacher_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    return answer.strip()


teacher_test_question = (
    "What is the primary function of the glomerulus in the kidney?"
)

teacher_test_answer = ask_teacher(
    teacher_test_question,
    max_new_tokens=128,
)

print("Question:")
print(teacher_test_question)

print("\nTeacher response:")
print(teacher_test_answer)

Question:
What is the primary function of the glomerulus in the kidney?

Teacher response:
The primary function of the glomerulus in the kidney is to filter blood and remove waste products and excess substances from the bloodstream. This process is a crucial part of the renal filtration system, which helps maintain the body's fluid balance and regulate blood pressure.

Here’s a more detailed explanation:

1. **Filtration**: The glomerulus is a network of tiny blood vessels (capillaries) within the kidney. Blood enters the glomerulus through the afferent arteriole and leaves through the efferent arteriole. As blood passes through the glomerulus, water and small molecules such as ions, glucose


**7.6.2 Pilot Synthetic Q&A Generation with the Teacher Model**

The teacher model has passed the inference sanity check. We now test its ability to construct supervised training examples from the same 10 passages previously evaluated with Llama 3.2 3B.

Using the same source passages allows us to compare the two generation approaches without changing the underlying data.

At this stage, the teacher model is instructed to:

- create exactly one educational question per usable passage;
- focus on the main nephrology concept present in the passage;
- ground the answer strictly in the supplied passage;
- avoid introducing outside medical information; and
- return `SKIP` only when the selected passage cannot support a coherent nephrology teaching example.

The raw teacher outputs will be retained for inspection before we scale generation to the full candidate corpus.

**Expected outcome:** the teacher model should produce more consistently focused nephrology questions and answers than the initial 3B-model pilot.

**7.6.3 Parse and Validate Teacher-Generated Examples**

The teacher model has generated a 50-example pilot dataset.

We now convert the raw teacher outputs into structured instruction-response records.

This step performs **structural validation only**. It does not attempt to determine medical correctness automatically.

A record is considered structurally valid when it contains:

- an `INSTRUCTION:` field;
- a `RESPONSE:` field;
- non-empty instruction and response text;
- no explicit `SKIP`; and
- no obvious meta-instruction in place of the actual question.

The original teacher output is retained for auditability.

**Expected outcome:** quantify how many pilot examples are structurally usable before scaling the generation process.

In [25]:
# Parse and structurally validate the completed Qwen pilot.

import re
import pandas as pd


def parse_teacher_output(text):
    text = str(text).strip()

    # Explicitly skipped by the teacher
    if text.upper() == "SKIP":
        return {
            "status": "SKIP",
            "instruction_parsed": "",
            "response_parsed": "",
        }

    # Extract the instruction and response fields
    match = re.search(
        r"INSTRUCTION:\s*(.*?)\s*RESPONSE:\s*(.*)",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )

    if not match:
        return {
            "status": "INVALID",
            "instruction_parsed": "",
            "response_parsed": "",
        }

    instruction = match.group(1).strip()
    response = match.group(2).strip()

    # Reject empty fields
    if not instruction or not response:
        return {
            "status": "INVALID",
            "instruction_parsed": instruction,
            "response_parsed": response,
        }

    # Reject obvious meta-instructions
    meta_patterns = [
        r"^create (one|a|an) ",
        r"^write (one|a|an) ",
        r"^generate ",
        r"^answer the question",
        r"^provide ",
    ]

    is_meta_instruction = any(
        re.search(pattern, instruction, flags=re.IGNORECASE)
        for pattern in meta_patterns
    )

    if is_meta_instruction:
        return {
            "status": "INVALID",
            "instruction_parsed": instruction,
            "response_parsed": response,
        }

    return {
        "status": "VALID",
        "instruction_parsed": instruction,
        "response_parsed": response,
    }


parsed_records = []

for _, row in teacher_pilot_df.iterrows():

    parsed = parse_teacher_output(
        row["teacher_output"]
    )

    parsed_records.append({
        "id": row["id"],
        "title": row["title"],
        "teacher_output": row["teacher_output"],
        **parsed,
    })


teacher_pilot_parsed_df = pd.DataFrame(parsed_records)


print("Pilot examples:", len(teacher_pilot_parsed_df))

print("\nStatus distribution:")
print(
    teacher_pilot_parsed_df["status"]
    .value_counts()
)

print("\nStructurally valid examples:")
print(
    teacher_pilot_parsed_df[
        teacher_pilot_parsed_df["status"] == "VALID"
    ][
        ["id", "title", "instruction_parsed", "response_parsed"]
    ].to_string(index=False)
)

Pilot examples: 50

Status distribution:
status
INVALID    36
VALID      12
SKIP        2
Name: count, dtype: int64

Structurally valid examples:
                        id                title                                                                                                       instruction_parsed                                                                                                                                                                                                               response_parsed
      Physiology_Levy_4128      Physiology_Levy                                              What effect does cortisol have on the action of antidiuretic hormone (ADH)?                                                                                                                        Cortisol inhibits the secretion and action of antidiuretic hormone (ADH), making it an ADH antagonist.
 InternalMed_Harrison_3651 InternalMed_Harrison                         

In [28]:
# Reset the teacher pilot cleanly without running model inference.

teacher_pilot_df = pilot_df.sample(
    n=min(10, len(pilot_df)),
    random_state=42
).copy()

teacher_pilot_df["teacher_output"] = ""

print("Fresh pilot passages:", len(teacher_pilot_df))
print("Teacher outputs initialized:", teacher_pilot_df["teacher_output"].eq("").sum())

Fresh pilot passages: 10
Teacher outputs initialized: 10


**7.6.4 Generate the Optimized Teacher Pilot**

We now run a small optimized pilot using Qwen2.5-7B-Instruct.

The pilot uses only 10 passages so that the generation quality can be evaluated before scaling to the full training corpus.

Compared with the earlier pilot, the generation budget is reduced because the desired output is intentionally concise:

```text
INSTRUCTION: <one educational question>
RESPONSE: <one grounded answer>

**7.6.5 Batch Teacher-Model Generation**

Sequential generation is relatively slow because each passage requires a separate model-generation call.

We now test batched generation, where multiple source passages are tokenized and processed together in a single model call.

Batching should improve GPU utilization and reduce the overhead of repeatedly invoking the teacher model.

The pilot remains intentionally small so that we can verify:

- batched tokenization works correctly;
- Qwen handles multiple prompts in one generation call;
- each generated response remains associated with the correct source passage; and
- the improved nephrology-focused prompt produces usable examples.

We continue to use deterministic generation and a maximum output length of 96 tokens.

**Expected outcome:** generate the 10 pilot examples more efficiently while preserving one output per source passage.

In [38]:
# Configure left-padding for batched generation with Qwen.

teacher_tokenizer.padding_side = "left"

# Qwen may not have a dedicated padding token.
# Use EOS as the padding token for generation.
if teacher_tokenizer.pad_token is None:
    teacher_tokenizer.pad_token = teacher_tokenizer.eos_token

print("Padding side:", teacher_tokenizer.padding_side)
print("Pad token:", teacher_tokenizer.pad_token)
print("Pad token ID:", teacher_tokenizer.pad_token_id)

Padding side: left
Pad token: <|endoftext|>
Pad token ID: 151643


In [39]:
# Corrected batched Qwen teacher generation.

import torch


# Build prompts
pilot_prompts = [
    create_teacher_training_example_prompt(passage)
    for passage in teacher_pilot_df["content_clean"]
]


# Apply Qwen's native chat template
formatted_prompts = []

for prompt in pilot_prompts:
    messages = [
        {
            "role": "system",
            "content": (
                "You generate high-quality, concise, grounded "
                "nephrology educational training examples."
            ),
        },
        {
            "role": "user",
            "content": prompt,
        },
    ]

    formatted_prompts.append(
        teacher_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    )


# Tokenize as a left-padded batch
inputs = teacher_tokenizer(
    formatted_prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=2048,
).to(teacher_model.device)


print("Batch size:", len(formatted_prompts))
print("Input tensor shape:", inputs["input_ids"].shape)


# Generate
with torch.inference_mode():
    outputs = teacher_model.generate(
        **inputs,
        max_new_tokens=96,
        do_sample=False,
        pad_token_id=teacher_tokenizer.pad_token_id,
    )


# Decode only the newly generated tokens.
# Because padding is on the LEFT, every row has the same
# input tensor width, so we remove that common width.
input_width = inputs["input_ids"].shape[1]

teacher_outputs = []

for i in range(len(formatted_prompts)):
    generated_tokens = outputs[i][input_width:]

    response = teacher_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    teacher_outputs.append(response)


# Store results
teacher_pilot_df["teacher_output"] = teacher_outputs


print("\nGenerated examples:", len(teacher_pilot_df))

for _, row in teacher_pilot_df.iterrows():
    print("\n" + "=" * 100)
    print(f"SOURCE ID: {row['id']}")
    print(f"SOURCE: {row['title']}")
    print("\nTEACHER OUTPUT:")
    print(row["teacher_output"])

Batch size: 10
Input tensor shape: torch.Size([10, 585])

Generated examples: 10

SOURCE ID: InternalMed_Harrison_3459
SOURCE: InternalMed_Harrison

TEACHER OUTPUT:
INSTRUCTION: What effect do glucocorticoids have on antidiuretic hormone (ADH) release, and how does this relate to euvolemic hyponatremia in secondary adrenal insufficiency?
RESPONSE: Glucocorticoids exert a negative feedback on ADH release by the posterior pituitary. In patients with secondary adrenal insufficiency, hydrocortisone replacement reduces circulating ADH, thereby normalizing the ADH response to

SOURCE ID: InternalMed_Harrison_22081
SOURCE: InternalMed_Harrison

TEACHER OUTPUT:
INSTRUCTION: What characteristic imaging finding is associated with analgesic nephropathy?
RESPONSE: Small, scarred kidneys with papillary calcifications best appreciated by computed tomography.

SOURCE ID: InternalMed_Harrison_11987
SOURCE: InternalMed_Harrison

TEACHER OUTPUT:
INSTRUCTION: What is the recommended antimicrobial regimen

**7.6.6 Prepare the Full Teacher-Generation Dataset**

The teacher-model pilot established that Qwen2.5-7B-Instruct can generate coherent nephrology educational examples when used with left-padded batched inference.

We now prepare the full set of cleaned nephrology passages for synthetic instruction-response generation.

Each source passage will produce at most one training example. Passages for which the teacher determines that a strong nephrology teaching example cannot be constructed will be marked `SKIP`.

The generation process will be performed in small batches and saved incrementally. This protects the experiment from losing already-generated examples if the Kaggle session is interrupted.

The raw teacher output will be retained together with the source identifier so that every generated example remains traceable to its originating passage.

**Expected outcome:** create a clean generation queue containing the selected nephrology passages that will be processed by the Qwen teacher model.

In [43]:
# Prepare the full teacher-generation queue.

teacher_generation_df = clean_df[
    [
        "id",
        "title",
        "content_clean",
    ]
].copy()

# Reset row numbering for deterministic batching.
teacher_generation_df.reset_index(drop=True, inplace=True)

# Add fields that will be populated during generation.
teacher_generation_df["teacher_output"] = ""
teacher_generation_df["generation_status"] = "PENDING"

print("Teacher-generation queue:", len(teacher_generation_df))

print("\nSource distribution:")
print(
    teacher_generation_df["title"]
    .value_counts()
    .sort_index()
)

print("\nGeneration status:")
print(
    teacher_generation_df["generation_status"]
    .value_counts()
)

Teacher-generation queue: 3113

Source distribution:
title
Anatomy_Gray              42
InternalMed_Harrison    2021
Pathology_Robbins        282
Pathoma_Husain            58
Pharmacology_Katzung     431
Physiology_Levy          279
Name: count, dtype: int64

Generation status:
generation_status
PENDING    3113
Name: count, dtype: int64


**7.6.7 Generate Synthetic Q&A with the Qwen Teacher**

We now generate the synthetic instruction-response dataset using
Qwen2.5-7B-Instruct.

The generation is performed in small batches rather than one passage
at a time. This improves GPU utilization while keeping memory usage
manageable on the Kaggle T4 GPU.

Each batch is saved immediately after generation. This creates a
checkpointed generation process so that completed examples are not
lost if the notebook session is interrupted.

For each source passage, the teacher produces either:

- one `INSTRUCTION` / `RESPONSE` pair; or
- `SKIP` when the passage does not support a sufficiently focused
  nephrology teaching example.

The raw teacher output is retained for later parsing and quality
validation.

### Generation configuration

- Teacher model: Qwen2.5-7B-Instruct
- Quantization: 4-bit NF4
- Generation: deterministic (`do_sample=False`)
- Batch size: 16
- Maximum generated tokens: 128
- Incremental checkpointing: enabled

**Expected outcome:** progressively generate and save the synthetic
nephrology training corpus while preserving intermediate checkpoints.

In [ ]:
# Generate the full synthetic training corpus in checkpointed batches.

import os
import torch
import pandas as pd

BATCH_SIZE = 16
MAX_NEW_TOKENS = 128

checkpoint_dir = "/kaggle/working/nephrollm_teacher_batches"
os.makedirs(checkpoint_dir, exist_ok=True)

total_rows = len(teacher_generation_df)

print("Total passages:", total_rows)
print("Batch size:", BATCH_SIZE)
print("Expected batches:", (total_rows + BATCH_SIZE - 1) // BATCH_SIZE)
print("Checkpoint directory:", checkpoint_dir)


def generate_teacher_batch(passages):
    """Generate one teacher response for each passage in a batch."""

    formatted_prompts = []

    for passage in passages:

        prompt = create_teacher_training_example_prompt(
            passage
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "You generate high-quality, concise, grounded "
                    "nephrology educational training examples."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ]

        formatted_prompts.append(
            teacher_tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        )

    # Tokenize as a left-padded batch
    inputs = teacher_tokenizer(
        formatted_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(teacher_model.device)

    with torch.inference_mode():
        outputs = teacher_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=teacher_tokenizer.pad_token_id,
        )

    # Because the tokenizer uses left padding, all rows have
    # the same input width.
    input_width = inputs["input_ids"].shape[1]

    responses = []

    for i in range(len(passages)):

        generated_tokens = outputs[i][input_width:]

        response = teacher_tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True,
        ).strip()

        responses.append(response)

    return responses


# Process batches sequentially.
for start_idx in range(0, total_rows, BATCH_SIZE):

    end_idx = min(
        start_idx + BATCH_SIZE,
        total_rows,
    )

    batch_number = start_idx // BATCH_SIZE + 1

    checkpoint_path = os.path.join(
        checkpoint_dir,
        f"teacher_batch_{batch_number:04d}.csv",
    )

    # Skip an already completed checkpoint.
    if os.path.exists(checkpoint_path):
        print(
            f"Batch {batch_number}: checkpoint already exists — skipping."
        )
        continue

    batch_df = teacher_generation_df.iloc[
        start_idx:end_idx
    ].copy()

    print(
        f"\nGenerating batch {batch_number} "
        f"({start_idx + 1}-{end_idx} of {total_rows})..."
    )

    try:

        batch_df["teacher_output"] = generate_teacher_batch(
            batch_df["content_clean"].tolist()
        )

        batch_df["generation_status"] = "GENERATED"

        batch_df.to_csv(
            checkpoint_path,
            index=False,
        )

        print(
            f"Batch {batch_number} saved: "
            f"{checkpoint_path}"
        )

    except Exception as e:

        print(
            f"Batch {batch_number} failed: {type(e).__name__}: {e}"
        )

        raise

print("\nTeacher generation completed for all batches.")

Total passages: 3113
Batch size: 16
Expected batches: 195
Checkpoint directory: /kaggle/working/nephrollm_teacher_batches

Generating batch 1 (1-16 of 3113)...
Batch 1 saved: /kaggle/working/nephrollm_teacher_batches/teacher_batch_0001.csv

Generating batch 2 (17-32 of 3113)...
Batch 2 saved: /kaggle/working/nephrollm_teacher_batches/teacher_batch_0002.csv

Generating batch 3 (33-48 of 3113)...
Batch 3 saved: /kaggle/working/nephrollm_teacher_batches/teacher_batch_0003.csv

Generating batch 4 (49-64 of 3113)...
Batch 4 saved: /kaggle/working/nephrollm_teacher_batches/teacher_batch_0004.csv

Generating batch 5 (65-80 of 3113)...
Batch 5 saved: /kaggle/working/nephrollm_teacher_batches/teacher_batch_0005.csv

Generating batch 6 (81-96 of 3113)...
Batch 6 saved: /kaggle/working/nephrollm_teacher_batches/teacher_batch_0006.csv

Generating batch 7 (97-112 of 3113)...
Batch 7 saved: /kaggle/working/nephrollm_teacher_batches/teacher_batch_0007.csv

Generating batch 8 (113-128 of 3113)...
Batc

# 7.7 Check for Evaluation Leakage

Before finalizing the training dataset, we perform an explicit leakage check against the NephroLLM evaluation benchmark.

The evaluation benchmark contains:

- **636 primary-evaluation questions**
- **160 held-out questions**

None of these evaluation records should be intentionally present in the QLoRA training data.

### Leakage checks

We will compare the candidate training examples against the evaluation benchmark using available identifying information and normalized text.

The check will look for:

- exact question matches;
- exact answer or explanation matches where applicable;
- duplicated or near-duplicated question text; and
- obvious overlap between source-derived training examples and evaluation items.

### Why this matters

A fine-tuned model could appear to improve simply because it has been exposed to evaluation questions or their answers during training.

That would make the before-versus-after comparison difficult to interpret.

The leakage check therefore acts as a quality-control step before the final training dataset is created.

### Evaluation boundary

```text
Training corpus
      │
      ├── Allowed → independent nephrology educational content
      │
      └── Excluded → evaluation benchmark content

Evaluation benchmark
      │
      ├── 636 primary questions
      └── 160 held-out questions

**7.7.1 validate all 195 checkpoints before parsing anything.**

In [45]:
### Validate all teacher-generation checkpoints

import os
import glob
import pandas as pd

checkpoint_dir = "/kaggle/working/nephrollm_teacher_batches"

files = sorted(
    glob.glob(
        os.path.join(
            checkpoint_dir,
            "teacher_batch_*.csv"
        )
    )
)

# Extract batch numbers
batch_numbers = sorted(
    int(
        os.path.basename(f)
        .split("_")[-1]
        .replace(".csv", "")
    )
    for f in files
)

expected_batches = set(range(1, 196))
existing_batches = set(batch_numbers)

missing_batches = sorted(
    expected_batches - existing_batches
)

unexpected_batches = sorted(
    existing_batches - expected_batches
)

print("Teacher checkpoint validation")
print("=" * 60)

print(f"Checkpoint files found : {len(files)}")
print(f"Expected checkpoint files: 195")
print(f"First batch            : {batch_numbers[0] if batch_numbers else 'None'}")
print(f"Last batch             : {batch_numbers[-1] if batch_numbers else 'None'}")

print("\nMissing batches:")
print(missing_batches if missing_batches else "None")

print("\nUnexpected batches:")
print(unexpected_batches if unexpected_batches else "None")


# Validate rows and statuses
total_rows = 0
all_ids = []
status_counts = {}

for file_path in files:
    df = pd.read_csv(file_path)

    total_rows += len(df)

    all_ids.extend(df["id"].astype(str).tolist())

    for status, count in df["generation_status"].value_counts().items():
        status_counts[status] = (
            status_counts.get(status, 0) + count
        )


duplicate_ids = (
    pd.Series(all_ids)
    .value_counts()
)

duplicate_ids = duplicate_ids[
    duplicate_ids > 1
]


print("\nRow validation")
print("=" * 60)

print(f"Total rows              : {total_rows}")
print(f"Expected rows           : 3113")

print("\nGeneration status:")
print(status_counts)

print(f"\nUnique source IDs       : {len(set(all_ids))}")
print(f"Duplicate source IDs    : {len(duplicate_ids)}")

if not duplicate_ids.empty:
    print("\nDuplicate IDs:")
    print(duplicate_ids.head(20))


# Final validation summary
validation_passed = (
    len(files) == 195
    and not missing_batches
    and not unexpected_batches
    and total_rows == 3113
    and len(set(all_ids)) == 3113
    and not duplicate_ids.size
)

print("\n" + "=" * 60)

if validation_passed:
    print("✅ CHECKPOINT VALIDATION PASSED")
    print("All 195 batches and 3,113 source passages are accounted for.")
else:
    print("⚠️ CHECKPOINT VALIDATION REQUIRES REVIEW")

Teacher checkpoint validation
Checkpoint files found : 195
Expected checkpoint files: 195
First batch            : 1
Last batch             : 195

Missing batches:
None

Unexpected batches:
None

Row validation
Total rows              : 3113
Expected rows           : 3113

Generation status:
{'GENERATED': 3113}

Unique source IDs       : 3113
Duplicate source IDs    : 0

✅ CHECKPOINT VALIDATION PASSED
All 195 batches and 3,113 source passages are accounted for.


**7.7.2 Combine Teacher-Generated Checkpoints**

The teacher-generation stage produced 195 checkpoint files covering all 3,113 selected nephrology passages.

The checkpoint validation confirmed that every expected batch is present and that each source passage appears exactly once.

We now combine these checkpoints into a single dataframe for downstream processing.

The original teacher output will be preserved unchanged at this stage. Parsing, quality filtering, deduplication, and evaluation-leakage checks will be performed on derived columns so that the raw generated output remains available for auditability.

**Expected outcome:** one master dataframe containing all 3,113 teacher-generated records with their source metadata and raw Qwen outputs.

In [46]:
### Combine all validated teacher-generation checkpoints

checkpoint_files = sorted(
    glob.glob(
        os.path.join(
            checkpoint_dir,
            "teacher_batch_*.csv"
        )
    )
)

teacher_generated_df = pd.concat(
    [
        pd.read_csv(file_path)
        for file_path in checkpoint_files
    ],
    ignore_index=True
)

print("Combined teacher-generation dataset")
print("=" * 60)

print("Rows:", len(teacher_generated_df))
print("Columns:", teacher_generated_df.columns.tolist())

print("\nSource distribution:")
print(
    teacher_generated_df["title"]
    .value_counts()
    .sort_index()
)

print("\nGeneration status:")
print(
    teacher_generated_df["generation_status"]
    .value_counts()
)

print("\nSample raw teacher output:")
print(
    teacher_generated_df[
        ["id", "title", "teacher_output"]
    ].head(3).to_string(index=False)
)

Combined teacher-generation dataset
Rows: 3113
Columns: ['id', 'title', 'content_clean', 'teacher_output', 'generation_status']

Source distribution:
title
Anatomy_Gray              42
InternalMed_Harrison    2021
Pathology_Robbins        282
Pathoma_Husain            58
Pharmacology_Katzung     431
Physiology_Levy          279
Name: count, dtype: int64

Generation status:
generation_status
GENERATED    3113
Name: count, dtype: int64

Sample raw teacher output:
              id        title                                                                                                                                                                                                                                                                                teacher_output
Anatomy_Gray_347 Anatomy_Gray                                                                                                                                                                                             

**7.7.3 Parse Teacher-Generated Q&A**

The 3,113 teacher-generated records have now been combined into a single dataframe.

Each `teacher_output` contains either:

- a structured `INSTRUCTION` / `RESPONSE` pair; or
- `SKIP` when the teacher determined that the source passage was not suitable for a strong nephrology example.

We now parse these outputs into separate structured columns.

This step is intentionally limited to **format parsing**. We will not yet judge the medical quality or nephrology relevance of the generated examples.

The original `teacher_output` column will be retained unchanged so that every generated example remains auditable against the raw teacher response.

**Expected outcome:** classify every generated record as `VALID`, `SKIP`, or `INVALID` and extract the instruction and response where available.

In [47]:
### Parse teacher-generated outputs

import re

parsed_df = teacher_generated_df.copy()

parsed_df["instruction"] = ""
parsed_df["response"] = ""
parsed_df["parse_status"] = "INVALID"


def parse_teacher_output(output):
    """
    Parse the expected teacher format:

    INSTRUCTION: <question>
    RESPONSE: <answer>

    or:

    SKIP
    """

    if pd.isna(output):
        return "", "", "INVALID"

    text = str(output).strip()

    # Exact SKIP output
    if text.upper() == "SKIP":
        return "", "", "SKIP"

    # Extract instruction and response
    match = re.search(
        r"INSTRUCTION:\s*(.*?)\s*RESPONSE:\s*(.*)",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )

    if not match:
        return "", "", "INVALID"

    instruction = match.group(1).strip()
    response = match.group(2).strip()

    if not instruction or not response:
        return "", "", "INVALID"

    return instruction, response, "VALID"


parsed_results = parsed_df["teacher_output"].apply(
    parse_teacher_output
)

parsed_df[
    ["instruction", "response", "parse_status"]
] = pd.DataFrame(
    parsed_results.tolist(),
    index=parsed_df.index,
)

print("Teacher-output parsing")
print("=" * 60)

print("\nParse status:")
print(
    parsed_df["parse_status"]
    .value_counts()
)

print("\nTotal records:", len(parsed_df))

print("\nSample parsed records:")
print(
    parsed_df[
        [
            "id",
            "parse_status",
            "instruction",
            "response",
        ]
    ].head(10).to_string(index=False)
)

Teacher-output parsing

Parse status:
parse_status
VALID    2888
SKIP      225
Name: count, dtype: int64

Total records: 3113

Sample parsed records:
              id parse_status                                                                                  instruction                                                                                                                                                                                                     response
Anatomy_Gray_347         SKIP                                                                                                                                                                                                                                                                                                          
Anatomy_Gray_460        VALID       Where does the hemiazygos vein typically arise and how does it connect to other veins?                               The hemiazygos vein typically arises at t

**7.7.4 Inspect Generated Q&A Quality by Source**

The parser identified 2,888 structurally valid instruction-response pairs.

A structurally valid pair is not necessarily suitable for the final nephrology training set. The teacher may generate a well-formed question from a passage whose central topic is actually outside nephrology.

We therefore first inspect the generated examples by their source textbook.

This diagnostic helps us understand where the generated Q&A is concentrated and whether certain source categories produce more peripheral examples.

No examples are removed in this step.

**Expected outcome:** quantify the valid training candidates by source and inspect representative examples before applying quality filters.

In [48]:
### Inspect valid teacher-generated examples by source

valid_teacher_df = parsed_df[
    parsed_df["parse_status"] == "VALID"
].copy()

print("Valid teacher-generated examples:", len(valid_teacher_df))

print("\nValid examples by source:")
print(
    valid_teacher_df["title"]
    .value_counts()
    .sort_index()
)

# Show a small random sample from each source
sample_per_source = 5

for source in sorted(valid_teacher_df["title"].unique()):

    source_df = valid_teacher_df[
        valid_teacher_df["title"] == source
    ]

    sample = source_df.sample(
        n=min(sample_per_source, len(source_df)),
        random_state=42
    )

    print("\n" + "=" * 100)
    print(f"SOURCE: {source}")
    print("=" * 100)

    for _, row in sample.iterrows():
        print(f"\nID: {row['id']}")
        print(f"INSTRUCTION: {row['instruction']}")
        print(f"RESPONSE: {row['response']}")
        print("-" * 100)

Valid teacher-generated examples: 2888

Valid examples by source:
title
Anatomy_Gray              29
InternalMed_Harrison    1891
Pathology_Robbins        268
Pathoma_Husain            47
Pharmacology_Katzung     391
Physiology_Levy          262
Name: count, dtype: int64

SOURCE: Anatomy_Gray

ID: Anatomy_Gray_2003
INSTRUCTION: What surgical procedure is used to create a high-flow system for dialysis?
RESPONSE: The radial artery is anastomosed to the cephalic vein at the wrist, or the brachial artery is anastomosed to the cephalic vein at the elbow.
----------------------------------------------------------------------------------------------------

ID: Anatomy_Gray_925
INSTRUCTION: What unique characteristic of renal cell tumors makes them distinct from other types of tumors?
RESPONSE: Renal cell tumors grow outward from the kidney, invading the fat and fascia, and spread into the renal vein, which is rare for other types of tumors.
----------------------------------------------------

**7.7.5 Identify Potentially Peripheral Training Examples**

The generated Q&A inspection shows that some structurally valid examples are strongly nephrology-focused, while others are only indirectly related to renal medicine.

We therefore introduce a conservative quality-screening step.

The screen will flag examples whose question and answer contain little or no explicit renal/nephrology terminology. It will be used for **diagnostic review first**, not automatic deletion.

This approach avoids discarding valid examples simply because they come from a broad medical textbook.

Examples that clearly teach renal physiology, kidney disease, renal pathology, electrolyte/acid-base mechanisms, dialysis, transplantation, or related renal concepts should remain eligible.

**Expected outcome:** identify potentially peripheral examples for review while preserving the complete parsed dataset.

In [49]:
### Flag potentially peripheral teacher-generated examples

nephro_terms = [
    "kidney",
    "renal",
    "nephro",
    "nephron",
    "glomerul",
    "glomerular",
    "glomerulonephritis",
    "nephritis",
    "nephrotic",
    "nephropathy",
    "tubular",
    "tubulointerstitial",
    "dialysis",
    "hemodialysis",
    "haemodialysis",
    "peritoneal dialysis",
    "transplant",
    "gfr",
    "glomerular filtration",
    "creatinine",
    "urea",
    "proteinuria",
    "hematuria",
    "haematuria",
    "oliguria",
    "anuria",
    "polyuria",
    "electrolyte",
    "hyperkalemia",
    "hypokalemia",
    "hyponatremia",
    "hypernatremia",
    "acid-base",
    "metabolic acidosis",
    "metabolic alkalosis",
    "renal artery",
    "renal vein",
    "renin",
    "aldosterone",
    "raas",
    "macula densa",
    "podocyte",
    "mesangial",
    "glomerulus",
    "glomeruli",
    "ureter",
    "urinary",
    "urine",
    "nephrolithiasis",
    "kidney stone",
    "renal stone",
    "polycystic kidney",
    "renal failure",
    "renal insufficiency",
    "chronic kidney disease",
    "acute kidney injury",
]

term_pattern = "|".join(
    re.escape(term)
    for term in nephro_terms
)

quality_df = valid_teacher_df.copy()

quality_df["qa_text"] = (
    quality_df["instruction"].fillna("")
    + " "
    + quality_df["response"].fillna("")
).str.lower()

quality_df["has_nephro_signal"] = quality_df[
    "qa_text"
].str.contains(
    term_pattern,
    regex=True,
    na=False
)

print("Quality-screening diagnostics")
print("=" * 60)

print("Total VALID examples:", len(quality_df))

print(
    "Examples with explicit nephrology/renal signal:",
    quality_df["has_nephro_signal"].sum()
)

print(
    "Potentially peripheral examples:",
    (~quality_df["has_nephro_signal"]).sum()
)

print("\nPotentially peripheral examples by source:")
print(
    quality_df.loc[
        ~quality_df["has_nephro_signal"],
        "title"
    ].value_counts().sort_index()
)

Quality-screening diagnostics
Total VALID examples: 2888
Examples with explicit nephrology/renal signal: 2078
Potentially peripheral examples: 810

Potentially peripheral examples by source:
title
Anatomy_Gray              3
InternalMed_Harrison    537
Pathology_Robbins        48
Pathoma_Husain           18
Pharmacology_Katzung    148
Physiology_Levy          56
Name: count, dtype: int64


**7.7.5 Deduplicate Generated Q&A**

We'll keep all 2,888 structurally valid Q&A pairs and exclude only the 225 explicit SKIP records.

That gives us a clean, reproducible rule:

Training candidates = teacher outputs that successfully follow the required INSTRUCTION / RESPONSE format.

The potential topical noise is now a documented limitation rather than something we artificially optimize away.
*****************************

The teacher-generation stage produced 2,888 structurally valid instruction-response pairs.

We will now check for duplicate generated examples.

This is a data-integrity step rather than a medical-quality filter. We will retain every structurally valid example unless it is an exact duplicate of another generated question.

The original source passage and raw `teacher_output` will remain available for auditability.

**Expected outcome:** identify and remove only exact duplicate training questions, if any, before checking for overlap with the evaluation benchmark.

In [50]:
### Check and remove exact duplicate generated questions

training_candidates = valid_teacher_df.copy()

before_count = len(training_candidates)

# Normalize only for duplicate detection.
# The original instruction text remains unchanged.
training_candidates["instruction_normalized"] = (
    training_candidates["instruction"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

duplicate_mask = training_candidates[
    "instruction_normalized"
].duplicated(
    keep="first"
)

duplicate_count = duplicate_mask.sum()

training_candidates = training_candidates[
    ~duplicate_mask
].copy()

training_candidates.reset_index(drop=True, inplace=True)

print("Generated Q&A deduplication")
print("=" * 60)

print("Before deduplication:", before_count)
print("Exact duplicate questions:", duplicate_count)
print("After deduplication:", len(training_candidates))

print(
    "\nDuplicate examples removed:",
    duplicate_count
)

Generated Q&A deduplication
Before deduplication: 2888
Exact duplicate questions: 3
After deduplication: 2885

Duplicate examples removed: 3


**7.7.6 Check for Exact Evaluation Leakage**

The final training candidates currently contain 2,885 generated instruction-response examples.

Before using them for QLoRA, we check whether any generated training instruction exactly matches a question in the NephroLLM evaluation benchmark.

The benchmark contains:

- 636 primary-evaluation questions; and
- 160 held-out questions.

For this leakage check, only the **question text** is used as the reference. Benchmark answers, solutions, and explanations remain evaluation-only and are not introduced into the training data.

We normalize text only for comparison by:

- converting to lowercase;
- normalizing whitespace; and
- removing insignificant leading/trailing whitespace.

This detects exact question reuse while allowing differences in capitalization or spacing.

**Expected outcome:** identify any training instructions that exactly reproduce an evaluation question.

In [51]:
### Exact-match evaluation leakage check

import re
import pandas as pd

# Load the official evaluation benchmark
primary_eval_path = (
    "/kaggle/input/datasets/shantanuss/"
    "nejm-ai-nephrology-benchmark/"
    "nephrollm_primary_eval.csv"
)

heldout_eval_path = (
    "/kaggle/input/datasets/shantanuss/"
    "nejm-ai-nephrology-benchmark/"
    "nephrollm_heldout_eval.csv"
)

primary_eval_df = pd.read_csv(primary_eval_path)
heldout_eval_df = pd.read_csv(heldout_eval_path)

evaluation_df = pd.concat(
    [
        primary_eval_df.assign(eval_split="primary_eval"),
        heldout_eval_df.assign(eval_split="held_out"),
    ],
    ignore_index=True
)


def normalize_question(text):
    """Normalize text for exact-match comparison."""
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# Normalize training instructions
training_candidates["instruction_normalized"] = (
    training_candidates["instruction"]
    .apply(normalize_question)
)

# Normalize benchmark questions
evaluation_df["question_normalized"] = (
    evaluation_df["Question"]
    .apply(normalize_question)
)

# Build lookup of evaluation questions
evaluation_question_lookup = dict(
    zip(
        evaluation_df["question_normalized"],
        evaluation_df["BenchmarkID"]
    )
)

# Identify exact matches
training_candidates["exact_eval_match"] = (
    training_candidates["instruction_normalized"]
    .isin(evaluation_question_lookup)
)

exact_matches = training_candidates[
    training_candidates["exact_eval_match"]
].copy()

print("Exact-match leakage check")
print("=" * 60)

print("Training candidates:", len(training_candidates))
print("Evaluation questions:", len(evaluation_df))

print(
    "\nExact evaluation-question matches:",
    len(exact_matches)
)

if len(exact_matches) > 0:

    exact_matches["matched_benchmark_id"] = (
        exact_matches["instruction_normalized"]
        .map(evaluation_question_lookup)
    )

    print("\nPotential leakage examples:")
    print(
        exact_matches[
            [
                "id",
                "instruction",
                "matched_benchmark_id"
            ]
        ].to_string(index=False)
    )

else:
    print(
        "✅ No training instruction exactly matches "
        "an evaluation question."
    )

Exact-match leakage check
Training candidates: 2885
Evaluation questions: 796

Exact evaluation-question matches: 0
✅ No training instruction exactly matches an evaluation question.


**7.7.7 Check for Near-Duplicate Evaluation Leakage**

Exact matching found no training instruction that reproduced an evaluation question verbatim.

Because the teacher model can paraphrase source material, we now perform a near-duplicate similarity check between:

- the 2,885 training instructions; and
- the 796 evaluation questions.

The purpose is to identify training instructions that may express substantially the same question as an evaluation item despite differences in wording.

This is a screening step. High similarity does not automatically mean leakage because two questions can legitimately test the same nephrology concept.

Potential matches will therefore be inspected before any training example is removed.

**Expected outcome:** identify and review highly similar training/evaluation question pairs.

In [52]:
### Near-duplicate screening using TF-IDF cosine similarity

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

train_questions = (
    training_candidates["instruction"]
    .fillna("")
    .astype(str)
    .tolist()
)

eval_questions = (
    evaluation_df["Question"]
    .fillna("")
    .astype(str)
    .tolist()
)

# Fit one vectorizer across both collections so they share the
# same vocabulary and representation.
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    strip_accents="unicode",
)

combined_text = train_questions + eval_questions

tfidf_matrix = vectorizer.fit_transform(combined_text)

train_matrix = tfidf_matrix[:len(train_questions)]
eval_matrix = tfidf_matrix[len(train_questions):]

# Compare every training question against every evaluation question.
similarity_matrix = cosine_similarity(
    train_matrix,
    eval_matrix
)

# Find the highest-similarity evaluation question for each
# training instruction.
best_eval_index = similarity_matrix.argmax(axis=1)
best_similarity = similarity_matrix.max(axis=1)

near_duplicate_screen = training_candidates[
    [
        "id",
        "instruction",
        "response",
    ]
].copy()

near_duplicate_screen["best_eval_similarity"] = best_similarity
near_duplicate_screen["best_eval_index"] = best_eval_index

near_duplicate_screen["matched_benchmark_id"] = [
    evaluation_df.iloc[idx]["BenchmarkID"]
    for idx in best_eval_index
]

near_duplicate_screen["matched_eval_question"] = [
    evaluation_df.iloc[idx]["Question"]
    for idx in best_eval_index
]

# Sort from most similar to least similar.
near_duplicate_screen = near_duplicate_screen.sort_values(
    "best_eval_similarity",
    ascending=False
).reset_index(drop=True)

print("Near-duplicate screening")
print("=" * 60)

print("Training instructions:", len(train_questions))
print("Evaluation questions:", len(eval_questions))

print(
    "\nHighest similarity:",
    round(
        near_duplicate_screen["best_eval_similarity"].iloc[0],
        4
    )
)

print("\nTop 20 candidate pairs:")
print(
    near_duplicate_screen[
        [
            "id",
            "best_eval_similarity",
            "matched_benchmark_id",
            "instruction",
            "matched_eval_question",
        ]
    ].head(20).to_string(index=False)
)

Near-duplicate screening
Training instructions: 2885
Evaluation questions: 796

Highest similarity: 0.6496

Top 20 candidate pairs:
                        id  best_eval_similarity matched_benchmark_id                                                                                                                                  instruction                                                                                                      matched_eval_question
 InternalMed_Harrison_3664              0.649611     NEPHRO-EVAL-0265                                                                            What is the most likely cause of the hypokalemia in this patient?                                                               What is the most likely cause of this patient’s hypokalemia?
 InternalMed_Harrison_3561              0.648236     NEPHRO-EVAL-0270                                                                                  What is the most frequent underlying cause of hyp

**What the near-duplicate check tells us**

The highest similarity is only 0.6496, and the top pair is:

Training: “What is the most likely cause of the hypokalemia in this patient?”
Evaluation: “What is the most likely cause of this patient’s hypokalemia?”

That's clearly very similar wording, but this alone doesn't establish data leakage. The questions could independently be generated from the same general nephrology concept.

The second pair is similar for the same reason: both ask about the underlying cause of hyperkalemia.

Several other high-scoring pairs are actually not semantically the same question despite sharing terminology. For example, transplantation vs. hyperbaric oxygen, or minimal-change-disease clinical features vs. treatment. That shows why we shouldn't use a simple cosine threshold to automatically remove examples.

I would use a conservative rule

For this project, I'd document:

No exact question overlap was identified. TF-IDF screening produced no obvious near-duplicate evaluation questions; the highest lexical similarity was 0.65, with high-scoring pairs generally sharing clinical terminology but addressing different questions.

I would not remove the 2,885 training examples based on this result.

This is also consistent with our decision not to over-filter the synthetic dataset.

One important distinction

The two high-similarity pairs about hypokalemia are worth noting in the experiment log, because they are the closest matches:

InternalMed_Harrison_3664 ↔ NEPHRO-EVAL-0265: 0.6496
InternalMed_Harrison_3561 ↔ NEPHRO-EVAL-0270: 0.6482

But I would classify these as conceptual overlap, not demonstrated leakage, based on the evidence we have.

Therefore our training set remains

2,885 examples

with:

0 exact evaluation-question matches
no obvious near-duplicate leakage requiring removal
raw teacher outputs preserved
source passage retained
225 explicit SKIPs excluded
3 exact duplicate questions removed

We can now move to 7.8 — Prepare the Final QLoRA Dataset.

# 7.8 Prepare the Final QLoRA Dataset

The leakage checks are complete:

- no exact evaluation-question matches were found;
- no obvious near-duplicate evaluation leakage was identified;
- 2,885 structurally valid, deduplicated teacher-generated examples remain.

We now convert these examples into the conversational format expected by Llama 3.2 3B Instruct.

Each example will contain:

- a system instruction defining the educational nephrology role;
- the generated educational question as the user message; and
- the teacher-generated answer as the assistant response.

The original source passage, source textbook, and raw teacher output will remain available as metadata for traceability, but they will not be included as additional conversational turns.

This produces the dataset that will be passed to the QLoRA fine-tuning pipeline.

**Expected outcome:** a final supervised fine-tuning dataset containing 2,885 nephrology educational instruction-response examples in Llama-compatible chat format.

In [53]:
### Build the final Llama chat-format training dataset

final_training_df = training_candidates.copy()

SYSTEM_MESSAGE = (
    "You are an educational assistant specializing in nephrology. "
    "Provide clear, factual, educational explanations. "
    "Do not provide diagnosis or personalized medical advice."
)

def build_chat_messages(row):
    return [
        {
            "role": "system",
            "content": SYSTEM_MESSAGE,
        },
        {
            "role": "user",
            "content": row["instruction"],
        },
        {
            "role": "assistant",
            "content": row["response"],
        },
    ]


final_training_df["messages"] = final_training_df.apply(
    build_chat_messages,
    axis=1
)

print("Final QLoRA training dataset")
print("=" * 60)

print("Training examples:", len(final_training_df))

print("\nColumns:")
print(final_training_df.columns.tolist())

print("\nSample chat example:")
print(
    final_training_df.iloc[0]["messages"]
)

Final QLoRA training dataset
Training examples: 2885

Columns:
['id', 'title', 'content_clean', 'teacher_output', 'generation_status', 'instruction', 'response', 'parse_status', 'instruction_normalized', 'exact_eval_match', 'messages']

Sample chat example:
[{'role': 'system', 'content': 'You are an educational assistant specializing in nephrology. Provide clear, factual, educational explanations. Do not provide diagnosis or personalized medical advice.'}, {'role': 'user', 'content': 'Where does the hemiazygos vein typically arise and how does it connect to other veins?'}, {'role': 'assistant', 'content': 'The hemiazygos vein typically arises at the junction between the left ascending lumbar vein and the left subcostal vein, and it often has a connection to the left renal vein.'}]


**7.8.1 Finalize the Training Dataset Schema**

The training examples are now represented in the Llama chat format.

Before creating the Hugging Face training dataset, we remove temporary columns that were created only for leakage and deduplication diagnostics.

We retain the source identifier, source textbook, original passage, raw teacher output, generated instruction, generated response, and final chat messages.

Keeping the raw source and teacher output preserves traceability for later auditing, while only the `messages` field will be used as the conversational training input.

**Expected outcome:** a clean final training dataframe with only the fields needed for training and auditability.

In [54]:
### Finalize training dataframe

final_training_df = final_training_df[
    [
        "id",
        "title",
        "content_clean",
        "teacher_output",
        "instruction",
        "response",
        "messages",
    ]
].copy()

final_training_df.reset_index(drop=True, inplace=True)

print("Final training dataframe")
print("=" * 60)

print("Rows:", len(final_training_df))
print("Columns:", final_training_df.columns.tolist())

print("\nMissing values:")
print(
    final_training_df[
        [
            "id",
            "title",
            "content_clean",
            "teacher_output",
            "instruction",
            "response",
            "messages",
        ]
    ].isna().sum()
)

print("\nSample:")
print(
    final_training_df.iloc[0][
        [
            "id",
            "title",
            "instruction",
            "response",
        ]
    ].to_string()
)

Final training dataframe
Rows: 2885
Columns: ['id', 'title', 'content_clean', 'teacher_output', 'instruction', 'response', 'messages']

Missing values:
id                0
title             0
content_clean     0
teacher_output    0
instruction       0
response          0
messages          0
dtype: int64

Sample:
id                                              Anatomy_Gray_460
title                                               Anatomy_Gray
instruction    Where does the hemiazygos vein typically arise...
response       The hemiazygos vein typically arises at the ju...


**7.8.2 Save the Final Training Dataset**

The final training dataframe contains 2,885 deduplicated, leakage-checked instruction-response examples.

We now persist the dataset to the Kaggle working directory before proceeding to tokenization and QLoRA preparation.

Two representations will be saved:

- CSV: convenient for inspection and audit
- JSONL: preserves the nested `messages` structure used for conversational fine-tuning

The raw teacher output and source metadata are retained in both artifacts where applicable.

**Expected outcome:** durable training-data files are created in `/kaggle/working/`.

In [55]:
### Save the final training dataset

import json
import os

output_dir = "/kaggle/working/nephrollm_final_training_data"
os.makedirs(output_dir, exist_ok=True)

# 1. Save an audit-friendly CSV.
# Convert messages to JSON text so the nested structure is preserved.
csv_df = final_training_df.copy()

csv_df["messages"] = csv_df["messages"].apply(
    lambda x: json.dumps(x, ensure_ascii=False)
)

csv_path = os.path.join(
    output_dir,
    "nephrollm_final_training_dataset.csv"
)

csv_df.to_csv(
    csv_path,
    index=False
)

# 2. Save JSONL with the complete training/audit record.
jsonl_path = os.path.join(
    output_dir,
    "nephrollm_final_training_dataset.jsonl"
)

with open(jsonl_path, "w", encoding="utf-8") as f:
    for record in final_training_df.to_dict(
        orient="records"
    ):
        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            )
            + "\n"
        )

print("Final training dataset saved")
print("=" * 60)

print("Examples:", len(final_training_df))

print("\nCSV:")
print(csv_path)

print("\nJSONL:")
print(jsonl_path)

print("\nCSV size (MB):", round(
    os.path.getsize(csv_path) / (1024 ** 2), 2
))

print("JSONL size (MB):", round(
    os.path.getsize(jsonl_path) / (1024 ** 2), 2
))

Final training dataset saved
Examples: 2885

CSV:
/kaggle/working/nephrollm_final_training_data/nephrollm_final_training_dataset.csv

JSONL:
/kaggle/working/nephrollm_final_training_data/nephrollm_final_training_dataset.jsonl

CSV size (MB): 5.38
JSONL size (MB): 5.59


# 8. Fine-tune using QLoRA

# 8.1 Explain QLoRA configuration

# 8.2 Configure LoRA adapters


# 8.3 Configure training arguments


# 8.4 Train the model


# 8.5 Inspect training metrics

# 9. Save the LoRA adapter

# 9.1 Save adapter

# 9.2 Verify saved artifacts

# 9.3 Save tokenizer/configuration if required

# 10. Run the exact same evaluation on the fine-tuned model

# 10.1 Load the fine-tuned adapter

# 10.2 Reuse the same inference protocol

# 10.3 Reuse the same MCQ prompt

# 10.4 Reuse the same parser

# 10.5 Evaluate the same 636 questions


# 10.6 Aggregate results

# 11. Evaluate on held-out questions


# 11.1 Run inference on 160 held-out questions

# 11.2 Parse responses


# 11.3 Calculate held-out accuracy


# 11.4 Analyze unparsed responses

# 12. Compare base vs fine-tuned model


# 12.1 Overall accuracy comparison

# 12.2 Answered-only accuracy comparison


# 12.3 Unparsed-response comparison


# 12.4 Subject-level comparison


# 12.5 Error-pattern comparison


# 12.6 Held-out comparison

# 13. Document findings, limitations, and conclusions


# 13.1 Key findings


# 13.2 What improved


# 13.3 What did not improve

# 13.4 Limitations


# 13.5 Data leakage considerations


# 13.6 Conclusion 